#  数据预处理

## 三锚点

In [7]:
import os
import numpy as np
import pandas as pd

# 输入文件
file_list = [
    r'D:\Desktop\paper2\数据\应用初始数据\1试验.csv',
    r'D:\Desktop\paper2\数据\应用初始数据\2圆形.csv',
    r'D:\Desktop\paper2\数据\应用初始数据\3方形.csv',
    r'D:\Desktop\paper2\数据\应用初始数据\4凹凸.csv'
]

# 需要提取并处理的列
extract_cols = ['rangetime(ms)', 'range0(m)', 'range1(m)', 'range2(m)']

# 仅对这三列添加缺失
target_cols = ['range0(m)', 'range1(m)', 'range2(m)']

# 每个文件中三列各自的缺失比例
missing_rates = np.array([
    [0.05, 0.05, 0.05],   # 1试验
    [0.10, 0.05, 0.05],   # 2圆形
    [0.05, 0.05, 0.05],   # 3方形
    [0.15, 0.10, 0.15]    # 4凹凸
])

# 固定随机种子
np.random.seed(36)

# 输出文件夹
output_folder = r'D:\Desktop\missing_output'
os.makedirs(output_folder, exist_ok=True)

for f, file_path in enumerate(file_list):

    # 读取 CSV
    df = pd.read_csv(file_path, encoding='utf-8-sig')

    # 只提取需要的 4 列
    df = df[extract_cols].copy()

    n = len(df)

    # 对三列分别制造缺失
    for j, col in enumerate(target_cols):

        rate = missing_rates[f, j]
        n_missing = round(n * rate)

        # 当前列单独随机抽取缺失行
        idx = np.random.permutation(n)[:n_missing]

        # 置为缺失
        df.loc[idx, col] = np.nan

    # 输出文件名
    name = os.path.splitext(os.path.basename(file_path))[0]
    out_file = os.path.join(output_folder, f'{name}_missing.csv')

    # 保存为 CSV
    df.to_csv(out_file, index=False, encoding='utf-8-sig')

    print(f'已保存：{out_file}')


已保存：D:\Desktop\missing_output\1试验_missing.csv
已保存：D:\Desktop\missing_output\2圆形_missing.csv
已保存：D:\Desktop\missing_output\3方形_missing.csv
已保存：D:\Desktop\missing_output\4凹凸_missing.csv


In [4]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import math
import traceback
from pathlib import Path
from typing import Tuple, Optional, Dict, Any, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 0. Matplotlib config
# =========================================================
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# =========================================================
# 1. Paths and parameters
# =========================================================
DATA_PAIRS = [
    {
        "name": "1试验",
        "uwb": Path(r"D:\Desktop\missing_output\1试验_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\1.txt"),
    },
    {
        "name": "2圆形",
        "uwb": Path(r"D:\Desktop\missing_output\2圆形_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\2.txt"),
    },
    {
        "name": "3方形",
        "uwb": Path(r"D:\Desktop\missing_output\3方形_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\3.txt"),
    },
    {
        "name": "4凹凸",
        "uwb": Path(r"D:\Desktop\missing_output\4凹凸_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\4.txt"),
    },
]

OUTPUT_DIR = Path(r"D:\Desktop\missing_output\batch_output")
OUTPUT_SUMMARY_XLSX = OUTPUT_DIR / "uwb_rtk_batch_summary.xlsx"
OUTPUT_SCENE_INFO_XLSX = OUTPUT_DIR / "parking_scene_reference.xlsx"

ANCHORS_2D = {
    0: (0.00, 0.00),
    1: (14.95, 0.00),
    2: (18.51, 33.15),
}

UWB_FLIP_X = True

TIME_SHIFT_CANDIDATES_COARSE = np.arange(-10.0, 10.01, 0.2)
TIME_SHIFT_FINE_STEP = 0.02
TIME_SHIFT_FINE_HALF_WIDTH = 0.3

FIG_DPI = 220

MAX_SPEED_MPS = 3.0
PREDICTION_BLEND = 0.80
MIN_DT_FOR_VEL = 1e-6

SMOOTH_WIN = 5
SMOOTH_ENABLE = True

MAX_CROSS_RESIDUAL_FOR_ALIGNMENT = 1.0
MIN_ALIGNMENT_POINTS = 8
MAX_CROSS_RESIDUAL_FOR_ALIGNMENT_RELAXED = 2.5
MIN_ALIGNMENT_POINTS_RELAXED = 5

MAX_GAP_FILL_POINTS = 8

# =========================================================
# 2. Logging helpers
# =========================================================
def ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def append_log(log_path: Path, text: str) -> None:
    ensure_parent_dir(log_path)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(text.rstrip() + "\n")

def log_print(log_path: Path, text: str) -> None:
    print(text)
    append_log(log_path, text)

def to_num_series(s):
    return pd.to_numeric(s, errors="coerce")

def infer_rate(t: np.ndarray) -> float:
    t = np.asarray(t, dtype=float)
    if len(t) < 3:
        return float("nan")
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        return float("nan")
    return 1.0 / float(np.median(dt))

def finite_count_2cols(a, b) -> int:
    a1 = pd.to_numeric(pd.Series(a), errors="coerce")
    b1 = pd.to_numeric(pd.Series(b), errors="coerce")
    return int((a1.notna() & b1.notna()).sum())

def interp1_nan_safe(t_ref, v_ref, t_query):
    t_ref = np.asarray(t_ref, dtype=float)
    v_ref = np.asarray(v_ref, dtype=float)
    t_query = np.asarray(t_query, dtype=float)

    m = np.isfinite(t_ref) & np.isfinite(v_ref)
    if np.count_nonzero(m) < 2:
        return np.full_like(t_query, np.nan, dtype=float)

    t0 = t_ref[m]
    v0 = v_ref[m]
    order = np.argsort(t0)
    t0 = t0[order]
    v0 = v0[order]

    out = np.interp(t_query, t0, v0)
    out[(t_query < t0[0]) | (t_query > t0[-1])] = np.nan
    return out

def interp2_at_times(t_ref, x_ref, y_ref, t_query):
    xi = interp1_nan_safe(t_ref, x_ref, t_query)
    yi = interp1_nan_safe(t_ref, y_ref, t_query)
    return xi, yi

def moving_average_1d_preserve_nan(x: np.ndarray, window: int = 5) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if window < 2:
        return x.copy()
    if window % 2 == 0:
        window += 1

    y = x.copy()
    pad = window // 2
    for i in range(len(x)):
        lo = max(0, i - pad)
        hi = min(len(x), i + pad + 1)
        seg = x[lo:hi]
        if np.count_nonzero(np.isfinite(seg)) >= 1:
            y[i] = np.nanmean(seg)
        else:
            y[i] = np.nan
    return y

def smooth_xy(x: np.ndarray, y: np.ndarray, window: int = 5) -> Tuple[np.ndarray, np.ndarray]:
    return moving_average_1d_preserve_nan(x, window), moving_average_1d_preserve_nan(y, window)

def calc_rmse(x1, y1, x2, y2):
    x1 = np.asarray(x1, dtype=float)
    y1 = np.asarray(y1, dtype=float)
    x2 = np.asarray(x2, dtype=float)
    y2 = np.asarray(y2, dtype=float)

    m = np.isfinite(x1) & np.isfinite(y1) & np.isfinite(x2) & np.isfinite(y2)
    if np.count_nonzero(m) < 3:
        return np.inf
    err = np.sqrt((x1[m] - x2[m]) ** 2 + (y1[m] - y2[m]) ** 2)
    return float(np.sqrt(np.mean(err ** 2)))

def is_valid_number(v) -> bool:
    try:
        return bool(np.isfinite(float(v)))
    except Exception:
        return False

def is_valid_xy(x, y) -> bool:
    return is_valid_number(x) and is_valid_number(y)

# =========================================================
# 3. Load UWB CSV/XLSX
# =========================================================
def robust_load_uwb_csv_3ranges(file_path: Path) -> Tuple[np.ndarray, pd.DataFrame]:
    if not file_path.exists():
        raise FileNotFoundError(f"UWB input file not found: {file_path}")

    if file_path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(
            file_path,
            engine="python",
            encoding="utf-8",
            na_values=["null", "NULL", "", "NaN", "nan"],
            keep_default_na=True,
        )

    df.columns = [str(c).strip() for c in df.columns]

    time_col = None
    for c in ["rangetime(ms)", "rangetime", "time", "时间_s", "time_s", "t"]:
        if c in df.columns:
            time_col = c
            break
    if time_col is None:
        raise ValueError("未找到时间列，请至少包含 rangetime(ms) 或 时间_s。")

    required_cols = ["range0(m)", "range1(m)", "range2(m)"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"文件缺少必要列: {missing}")

    numeric_cols = [time_col, "range0(m)", "range1(m)", "range2(m)"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    t_raw = df[time_col].to_numpy(dtype=float)
    if np.isfinite(t_raw).sum() < 2:
        raise ValueError(f"{time_col} 列有效数据不足。")

    first_valid = t_raw[np.isfinite(t_raw)][0]
    if time_col == "rangetime(ms)" or "ms" in time_col.lower():
        t = (t_raw - first_valid) / 1000.0
    else:
        t = t_raw - first_valid

    idx = np.arange(len(t), dtype=float)
    m = np.isfinite(t)
    if np.count_nonzero(m) >= 2 and np.count_nonzero(~m) > 0:
        t[~m] = np.interp(idx[~m], idx[m], t[m])

    return t, df.copy()

# =========================================================
# 4. RTK parsing and global parking-lot scene system
# =========================================================
def nmea_degmin_to_deg(dm: str, hemi: str) -> float:
    if not dm:
        return float("nan")
    try:
        dm = float(dm)
    except Exception:
        return float("nan")

    deg = int(dm // 100)
    minutes = dm - deg * 100
    val = deg + minutes / 60.0
    if hemi in ("S", "W"):
        val = -val
    return val

def parse_gga_time_to_sec(tfield: str) -> Optional[float]:
    if not tfield:
        return None
    try:
        hh = int(tfield[0:2])
        mm = int(tfield[2:4])
        ss = float(tfield[4:])
        return hh * 3600 + mm * 60 + ss
    except Exception:
        return None

def make_time_monotonic(t_raw: np.ndarray) -> np.ndarray:
    t_raw = np.asarray(t_raw, dtype=float)
    if t_raw.size == 0:
        return t_raw.copy()

    t = np.array(t_raw, dtype=float)
    day_offset = 0.0
    prev = t[0]

    for i in range(1, len(t)):
        cur = t[i]
        while cur + day_offset < prev - 1.0:
            day_offset += 86400.0
        adj = cur + day_offset
        if adj < prev:
            adj = prev
        t[i] = adj
        prev = adj

    return t

def geodetic_to_local_xy(lat: float, lon: float, lat0: float, lon0: float) -> Tuple[float, float]:
    R = 6378137.0
    lat0r = math.radians(lat0)
    x = math.radians(lon - lon0) * math.cos(lat0r) * R
    y = math.radians(lat - lat0) * R
    return x, y

def parse_rtk_gga_raw(rtk_log_path: Path) -> pd.DataFrame:
    if not rtk_log_path.exists():
        raise FileNotFoundError(f"RTK file not found: {rtk_log_path}")

    times_raw = []
    lats_raw = []
    lons_raw = []

    with open(rtk_log_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not (line.startswith("$") and "GGA" in line):
                continue

            parts = line.split(",")
            if len(parts) < 7:
                continue

            fix = parts[6]
            if fix not in ("4", "5", "2", "1"):
                continue

            tsec = parse_gga_time_to_sec(parts[1] if len(parts) > 1 else "")
            lat = nmea_degmin_to_deg(parts[2], parts[3] if len(parts) > 3 else "")
            lon = nmea_degmin_to_deg(parts[4], parts[5] if len(parts) > 5 else "")

            if (tsec is None) or (not np.isfinite(lat)) or (not np.isfinite(lon)):
                continue

            times_raw.append(float(tsec))
            lats_raw.append(float(lat))
            lons_raw.append(float(lon))

    if len(times_raw) < 2:
        raise ValueError(f"RTK 文件有效 GGA 点不足: {rtk_log_path}")

    t = make_time_monotonic(np.array(times_raw, float))
    t = t - t[0]

    return pd.DataFrame({
        "时间_s": t,
        "纬度_deg": np.array(lats_raw, float),
        "经度_deg": np.array(lons_raw, float),
    })

def estimate_scene_rotation_from_points(x_all: np.ndarray, y_all: np.ndarray) -> float:
    pts = np.column_stack([x_all, y_all]).astype(float)
    m = np.isfinite(pts).all(axis=1)
    pts = pts[m]
    if len(pts) < 3:
        return 0.0

    pts0 = pts - pts.mean(axis=0, keepdims=True)
    cov = np.cov(pts0.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    v = eigvecs[:, np.argmax(eigvals)]  # 主方向

    theta = math.atan2(v[1], v[0])      # 主轴相对 East 的角度
    rot = -theta                        # 旋转到接近 X 正方向

    return float(rot)

def rotate_xy(x: np.ndarray, y: np.ndarray, theta_rad: float) -> Tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    c = math.cos(theta_rad)
    s = math.sin(theta_rad)
    xr = c * x - s * y
    yr = s * x + c * y
    return xr, yr

def build_global_parking_scene(data_pairs: List[Dict[str, Path]]) -> Dict[str, Any]:
    all_lat = []
    all_lon = []
    raw_dfs = {}

    for item in data_pairs:
        df_raw = parse_rtk_gga_raw(item["rtk"])
        raw_dfs[item["name"]] = df_raw

        all_lat.extend(df_raw["纬度_deg"].tolist())
        all_lon.extend(df_raw["经度_deg"].tolist())

    if len(all_lat) == 0:
        raise ValueError("全部 RTK 文件都没有有效经纬度，无法构建统一停车场场景坐标系。")

    lat0 = float(np.median(np.asarray(all_lat, dtype=float)))
    lon0 = float(np.median(np.asarray(all_lon, dtype=float)))

    all_x_enu = []
    all_y_enu = []

    for _, df_raw in raw_dfs.items():
        lat_arr = df_raw["纬度_deg"].to_numpy(dtype=float)
        lon_arr = df_raw["经度_deg"].to_numpy(dtype=float)
        xe = np.empty_like(lat_arr)
        yn = np.empty_like(lat_arr)
        for i in range(len(lat_arr)):
            xe[i], yn[i] = geodetic_to_local_xy(lat_arr[i], lon_arr[i], lat0, lon0)
        all_x_enu.append(xe)
        all_y_enu.append(yn)

    x_all = np.concatenate(all_x_enu)
    y_all = np.concatenate(all_y_enu)

    theta_scene = estimate_scene_rotation_from_points(x_all, y_all)

    x_scene_all, y_scene_all = rotate_xy(x_all, y_all, theta_scene)

    # 平移到正坐标，便于后续停车场区域划分
    scene_min_x = np.nanmin(x_scene_all)
    scene_min_y = np.nanmin(y_scene_all)
    tx = -scene_min_x
    ty = -scene_min_y

    return {
        "lat0": lat0,
        "lon0": lon0,
        "theta_scene_rad": theta_scene,
        "theta_scene_deg": math.degrees(theta_scene),
        "shift_x": tx,
        "shift_y": ty,
        "raw_dfs": raw_dfs,
    }

def read_rtk_series_to_scene(rtk_log_path: Path, scene_ref: Dict[str, Any]) -> pd.DataFrame:
    df_raw = parse_rtk_gga_raw(rtk_log_path)

    lat0 = scene_ref["lat0"]
    lon0 = scene_ref["lon0"]
    theta_scene = scene_ref["theta_scene_rad"]
    shift_x = scene_ref["shift_x"]
    shift_y = scene_ref["shift_y"]

    lat_arr = df_raw["纬度_deg"].to_numpy(dtype=float)
    lon_arr = df_raw["经度_deg"].to_numpy(dtype=float)

    E = np.empty_like(lat_arr)
    N = np.empty_like(lat_arr)
    for i in range(len(lat_arr)):
        E[i], N[i] = geodetic_to_local_xy(lat_arr[i], lon_arr[i], lat0, lon0)

    X_scene, Y_scene = rotate_xy(E, N, theta_scene)
    X_scene = X_scene + shift_x
    Y_scene = Y_scene + shift_y

    return pd.DataFrame({
        "时间_s": df_raw["时间_s"].to_numpy(dtype=float),
        "RTK_ENU_X_m": E,
        "RTK_ENU_Y_m": N,
        "RTK场景X_m": X_scene,
        "RTK场景Y_m": Y_scene,
        "纬度_deg": lat_arr,
        "经度_deg": lon_arr,
    })

# =========================================================
# 5. Geometry helpers
# =========================================================
def trilaterate_2d_three_anchors(anchors_xy: np.ndarray, d: np.ndarray) -> Tuple[float, float]:
    anchors_xy = np.asarray(anchors_xy, dtype=float)
    d = np.asarray(d, dtype=float)

    (x1, y1), (x2, y2), (x3, y3) = anchors_xy
    d1, d2, d3 = d

    A = np.array([
        [2 * (x2 - x1), 2 * (y2 - y1)],
        [2 * (x3 - x1), 2 * (y3 - y1)],
    ], dtype=float)

    b = np.array([
        (x2**2 - x1**2) + (y2**2 - y1**2) + (d1**2 - d2**2),
        (x3**2 - x1**2) + (y3**2 - y1**2) + (d1**2 - d3**2),
    ], dtype=float)

    if abs(np.linalg.det(A)) < 1e-12:
        return np.nan, np.nan

    try:
        sol = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        return np.nan, np.nan
    return float(sol[0]), float(sol[1])

def circle_circle_intersections(
    c0: Tuple[float, float],
    r0: float,
    c1: Tuple[float, float],
    r1: float
) -> List[Tuple[float, float]]:
    x0, y0 = c0
    x1, y1 = c1

    dx = x1 - x0
    dy = y1 - y0
    d = math.hypot(dx, dy)

    if not np.isfinite(d) or d < 1e-12:
        return []
    if d > r0 + r1:
        return []
    if d < abs(r0 - r1):
        return []
    if d == 0 and abs(r0 - r1) < 1e-12:
        return []

    a = (r0**2 - r1**2 + d**2) / (2 * d)
    h2 = r0**2 - a**2
    if h2 < -1e-10:
        return []
    h = math.sqrt(max(h2, 0.0))

    xm = x0 + a * dx / d
    ym = y0 + a * dy / d

    rx = -dy * (h / d)
    ry = dx * (h / d)

    p1 = (xm + rx, ym + ry)
    p2 = (xm - rx, ym - ry)

    if math.hypot(p1[0] - p2[0], p1[1] - p2[1]) < 1e-10:
        return [p1]
    return [p1, p2]

def project_point_to_circle(
    center: Tuple[float, float],
    radius: float,
    point: Tuple[float, float]
) -> Tuple[float, float]:
    cx, cy = center
    px, py = point

    vx = px - cx
    vy = py - cy
    norm = math.hypot(vx, vy)

    if norm < 1e-12:
        return cx + radius, cy
    scale = radius / norm
    return cx + vx * scale, cy + vy * scale

def compute_cross_residual_single(x: float, y: float, d: np.ndarray, anchors_xy: np.ndarray) -> float:
    d = np.asarray(d, dtype=float)
    anchors_xy = np.asarray(anchors_xy, dtype=float)

    if not (np.isfinite(x) and np.isfinite(y) and np.all(np.isfinite(d))):
        return np.nan

    errs = []
    for i in range(3):
        ax, ay = anchors_xy[i]
        pred = math.sqrt((x - ax) ** 2 + (y - ay) ** 2)
        errs.append(pred - d[i])

    errs = np.asarray(errs, dtype=float)
    return float(np.sqrt(np.mean(errs ** 2)))

# =========================================================
# 6. Weak-prior stage 1 trajectory reconstruction
# =========================================================
def build_prediction(
    t: np.ndarray,
    x_prev: Optional[float],
    y_prev: Optional[float],
    x_prev2: Optional[float],
    y_prev2: Optional[float],
    i: int,
) -> Tuple[float, float]:
    if not is_valid_xy(x_prev, y_prev):
        return np.nan, np.nan

    if not is_valid_xy(x_prev2, y_prev2) or i < 2:
        return float(x_prev), float(y_prev)

    dt1 = max(t[i - 1] - t[i - 2], MIN_DT_FOR_VEL)
    dt2 = max(t[i] - t[i - 1], MIN_DT_FOR_VEL)

    vx = (float(x_prev) - float(x_prev2)) / dt1
    vy = (float(y_prev) - float(y_prev2)) / dt1

    x_pred = float(x_prev) + PREDICTION_BLEND * vx * dt2
    y_pred = float(y_prev) + PREDICTION_BLEND * vy * dt2
    return x_pred, y_pred

def limit_jump(
    x_candidate: float,
    y_candidate: float,
    x_prev: Optional[float],
    y_prev: Optional[float],
    dt: float,
    max_speed_mps: float = MAX_SPEED_MPS
) -> Tuple[float, float]:
    if not is_valid_xy(x_candidate, y_candidate):
        return np.nan, np.nan

    if not is_valid_xy(x_prev, y_prev):
        return float(x_candidate), float(y_candidate)

    max_dist = max_speed_mps * max(dt, MIN_DT_FOR_VEL)
    dx = float(x_candidate) - float(x_prev)
    dy = float(y_candidate) - float(y_prev)
    dist = math.hypot(dx, dy)

    if dist <= max_dist or dist < 1e-12:
        return float(x_candidate), float(y_candidate)

    scale = max_dist / dist
    return float(x_prev) + dx * scale, float(y_prev) + dy * scale

def solve_stage1_with_weak_prior(
    t: np.ndarray,
    df_ranges: pd.DataFrame,
    anchors_xy: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    vals = df_ranges[["range0(m)", "range1(m)", "range2(m)"]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    n = len(vals)

    x = np.full(n, np.nan, dtype=float)
    y = np.full(n, np.nan, dtype=float)
    cross_residual = np.full(n, np.nan, dtype=float)
    source = np.full(n, "", dtype=object)

    prev_valid_indices: List[int] = []

    for i in range(n):
        row = vals[i]
        valid_idx = np.where(np.isfinite(row))[0]
        n_valid = len(valid_idx)

        x_prev = y_prev = x_prev2 = y_prev2 = None
        if len(prev_valid_indices) >= 1:
            j1 = prev_valid_indices[-1]
            x_prev, y_prev = float(x[j1]), float(y[j1])
        if len(prev_valid_indices) >= 2:
            j2 = prev_valid_indices[-2]
            x_prev2, y_prev2 = float(x[j2]), float(y[j2])

        x_pred, y_pred = build_prediction(t, x_prev, y_prev, x_prev2, y_prev2, i)

        if n_valid == 3:
            xi, yi = trilaterate_2d_three_anchors(anchors_xy, row)
            if is_valid_xy(xi, yi):
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)
                x[i], y[i] = xi, yi
                cross_residual[i] = compute_cross_residual_single(xi, yi, row, anchors_xy)
                source[i] = "direct_3range"
                prev_valid_indices.append(i)
            else:
                source[i] = "bad_3range"
            continue

        if n_valid == 2:
            a0, a1 = valid_idx.tolist()
            p0 = tuple(anchors_xy[a0])
            p1 = tuple(anchors_xy[a1])
            r0 = float(row[a0])
            r1 = float(row[a1])

            cands = circle_circle_intersections(p0, r0, p1, r1)
            if len(cands) > 0:
                if is_valid_xy(x_pred, y_pred):
                    dists = [math.hypot(cx - x_pred, cy - y_pred) for cx, cy in cands]
                    best_idx = int(np.argmin(dists))
                elif is_valid_xy(x_prev, y_prev):
                    dists = [math.hypot(cx - x_prev, cy - y_prev) for cx, cy in cands]
                    best_idx = int(np.argmin(dists))
                else:
                    best_idx = 0

                xi, yi = cands[best_idx]
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)
                x[i], y[i] = xi, yi
                source[i] = "prior_2range"
                prev_valid_indices.append(i)
            else:
                source[i] = "bad_2range"
            continue

        if n_valid == 1:
            a0 = int(valid_idx[0])
            p0 = tuple(anchors_xy[a0])
            r0 = float(row[a0])

            if is_valid_xy(x_pred, y_pred):
                xi, yi = project_point_to_circle(p0, r0, (x_pred, y_pred))
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)
                x[i], y[i] = xi, yi
                source[i] = "prior_1range"
                prev_valid_indices.append(i)
            elif is_valid_xy(x_prev, y_prev):
                xi, yi = project_point_to_circle(p0, r0, (x_prev, y_prev))
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)
                x[i], y[i] = xi, yi
                source[i] = "prior_1range"
                prev_valid_indices.append(i)
            else:
                source[i] = "none_1range_no_prior"
            continue

        source[i] = "none_0range"

    return x, y, cross_residual, source

# =========================================================
# 7. Fill short gaps for 0-range cases
# =========================================================
def fill_short_gaps_linear(
    t: np.ndarray,
    x: np.ndarray,
    y: np.ndarray,
    source: np.ndarray,
    max_gap_points: int = 8
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    x2 = np.asarray(x, dtype=float).copy()
    y2 = np.asarray(y, dtype=float).copy()
    source2 = source.copy()

    n = len(x2)
    i = 0
    while i < n:
        if np.isfinite(x2[i]) and np.isfinite(y2[i]):
            i += 1
            continue

        j = i
        while j < n and not (np.isfinite(x2[j]) and np.isfinite(y2[j])):
            j += 1

        gap_len = j - i
        left = i - 1
        right = j

        can_fill = (
            gap_len <= max_gap_points and
            left >= 0 and right < n and
            np.isfinite(x2[left]) and np.isfinite(y2[left]) and
            np.isfinite(x2[right]) and np.isfinite(y2[right])
        )

        if can_fill:
            for k in range(i, j):
                alpha = (t[k] - t[left]) / max(t[right] - t[left], MIN_DT_FOR_VEL)
                x2[k] = (1 - alpha) * x2[left] + alpha * x2[right]
                y2[k] = (1 - alpha) * y2[left] + alpha * y2[right]
                if source2[k] in ["none_0range", "none_1range_no_prior", "bad_2range", "bad_3range"]:
                    source2[k] = "interp_gap"
        i = j

    return x2, y2, source2

# =========================================================
# 8. Stage 2 smoothing
# =========================================================
def smooth_trajectory_stage2(
    x: np.ndarray,
    y: np.ndarray,
    source: np.ndarray,
    window: int = 5
) -> Tuple[np.ndarray, np.ndarray]:
    if not SMOOTH_ENABLE:
        return np.asarray(x, dtype=float).copy(), np.asarray(y, dtype=float).copy()

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x_s, y_s = smooth_xy(x, y, window=window)

    strong_mask = np.isin(source, ["direct_3range", "prior_2range"])
    x_s[strong_mask] = x[strong_mask]
    y_s[strong_mask] = y[strong_mask]

    return x_s, y_s

# =========================================================
# 9. Alignment
# =========================================================
def simple_flip_uwb(x: np.ndarray, y: np.ndarray, flip_x=True):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if flip_x:
        return -x, y
    return x.copy(), y.copy()

def kabsch_2d(P, Q):
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)

    Pc = P - P.mean(axis=0, keepdims=True)
    Qc = Q - Q.mean(axis=0, keepdims=True)

    H = Pc.T @ Qc
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    t = Q.mean(axis=0) - (R @ P.mean(axis=0))
    theta_deg = math.degrees(math.atan2(R[1, 0], R[0, 0]))
    return R, t, theta_deg

def apply_rigid_transform(x, y, R, t):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    XY = np.column_stack([x, y]).astype(float)
    XY2 = (R @ XY.T).T + t
    return XY2[:, 0], XY2[:, 1]

def build_alignment_mask_strict(uwb_x, uwb_y, cross_residual, source, rtk_xi, rtk_yi):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    cross_residual = np.asarray(cross_residual, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    strong_source_mask = np.isin(source, ["direct_3range", "prior_2range"])
    residual_ok = np.isfinite(cross_residual) & (cross_residual <= MAX_CROSS_RESIDUAL_FOR_ALIGNMENT)
    prior2_ok = (source == "prior_2range")

    return (
        np.isfinite(uwb_x) &
        np.isfinite(uwb_y) &
        np.isfinite(rtk_xi) &
        np.isfinite(rtk_yi) &
        strong_source_mask &
        (residual_ok | prior2_ok)
    )

def build_alignment_mask_relaxed(uwb_x, uwb_y, cross_residual, source, rtk_xi, rtk_yi):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    cross_residual = np.asarray(cross_residual, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    source_ok = np.isin(source, ["direct_3range", "prior_2range", "prior_1range", "interp_gap"])
    residual_ok = (~np.isfinite(cross_residual)) | (cross_residual <= MAX_CROSS_RESIDUAL_FOR_ALIGNMENT_RELAXED)

    return (
        np.isfinite(uwb_x) &
        np.isfinite(uwb_y) &
        np.isfinite(rtk_xi) &
        np.isfinite(rtk_yi) &
        source_ok &
        residual_ok
    )

def build_alignment_mask_fallback_allfinite(uwb_x, uwb_y, rtk_xi, rtk_yi):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    return (
        np.isfinite(uwb_x) &
        np.isfinite(uwb_y) &
        np.isfinite(rtk_xi) &
        np.isfinite(rtk_yi)
    )

def evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, mask):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    if np.count_nonzero(mask) < 3:
        return None

    P = np.column_stack([uwb_x[mask], uwb_y[mask]])
    Q = np.column_stack([rtk_xi[mask], rtk_yi[mask]])

    R, t, theta_deg = kabsch_2d(P, Q)
    uwb_x2, uwb_y2 = apply_rigid_transform(uwb_x, uwb_y, R, t)
    score = calc_rmse(uwb_x2[mask], uwb_y2[mask], rtk_xi[mask], rtk_yi[mask])

    return {
        "R": R,
        "t": t,
        "theta_deg": float(theta_deg),
        "rmse": float(score),
        "uwb_x_aligned": uwb_x2,
        "uwb_y_aligned": uwb_y2,
    }

def search_best_time_shift_for_alignment(
    uwb_t, uwb_x, uwb_y, cross_residual, source, rtk_t, rtk_x, rtk_y
):
    uwb_t = np.asarray(uwb_t, dtype=float)
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    cross_residual = np.asarray(cross_residual, dtype=float)
    rtk_t = np.asarray(rtk_t, dtype=float)
    rtk_x = np.asarray(rtk_x, dtype=float)
    rtk_y = np.asarray(rtk_y, dtype=float)

    best = None
    best_score = np.inf

    def _search(time_shift_candidates):
        nonlocal best, best_score

        for dt in time_shift_candidates:
            rtk_xi, rtk_yi = interp2_at_times(rtk_t + dt, rtk_x, rtk_y, uwb_t)

            candidate = None
            mode = None
            align_mask = None

            m_strict = build_alignment_mask_strict(
                uwb_x=uwb_x,
                uwb_y=uwb_y,
                cross_residual=cross_residual,
                source=source,
                rtk_xi=rtk_xi,
                rtk_yi=rtk_yi,
            )
            if np.count_nonzero(m_strict) >= MIN_ALIGNMENT_POINTS:
                out = evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, m_strict)
                if out is not None:
                    candidate = out
                    mode = "strict"
                    align_mask = m_strict

            if candidate is None:
                m_relaxed = build_alignment_mask_relaxed(
                    uwb_x=uwb_x,
                    uwb_y=uwb_y,
                    cross_residual=cross_residual,
                    source=source,
                    rtk_xi=rtk_xi,
                    rtk_yi=rtk_yi,
                )
                if np.count_nonzero(m_relaxed) >= MIN_ALIGNMENT_POINTS_RELAXED:
                    out = evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, m_relaxed)
                    if out is not None:
                        candidate = out
                        mode = "relaxed"
                        align_mask = m_relaxed

            if candidate is None:
                m_all = build_alignment_mask_fallback_allfinite(uwb_x, uwb_y, rtk_xi, rtk_yi)
                if np.count_nonzero(m_all) >= 3:
                    out = evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, m_all)
                    if out is not None:
                        candidate = out
                        mode = "fallback_allfinite"
                        align_mask = m_all

            if candidate is None:
                continue

            score = candidate["rmse"]
            if score < best_score:
                best_score = score
                best = {
                    "time_shift_s": float(dt),
                    "R": candidate["R"],
                    "t": candidate["t"],
                    "theta_deg": candidate["theta_deg"],
                    "rmse": candidate["rmse"],
                    "rtk_x_interp": rtk_xi,
                    "rtk_y_interp": rtk_yi,
                    "uwb_x_aligned": candidate["uwb_x_aligned"],
                    "uwb_y_aligned": candidate["uwb_y_aligned"],
                    "align_mask": align_mask,
                    "align_mode": mode,
                    "align_points": int(np.count_nonzero(align_mask)),
                }

    _search(TIME_SHIFT_CANDIDATES_COARSE)
    if best is None:
        return None

    c = best["time_shift_s"]
    fine_candidates = np.arange(
        c - TIME_SHIFT_FINE_HALF_WIDTH,
        c + TIME_SHIFT_FINE_HALF_WIDTH + TIME_SHIFT_FINE_STEP,
        TIME_SHIFT_FINE_STEP
    )
    _search(fine_candidates)
    return best

# =========================================================
# 10. Plot
# =========================================================
def plot_aligned_trajectory(rtk_x, rtk_y, uwb_x_aligned, uwb_y_aligned, output_path: Path):
    ensure_parent_dir(output_path)

    rtk_x = pd.to_numeric(pd.Series(rtk_x), errors="coerce")
    rtk_y = pd.to_numeric(pd.Series(rtk_y), errors="coerce")
    uwb_x_aligned = pd.to_numeric(pd.Series(uwb_x_aligned), errors="coerce")
    uwb_y_aligned = pd.to_numeric(pd.Series(uwb_y_aligned), errors="coerce")

    plt.figure(figsize=(8, 8))

    m_rtk = rtk_x.notna() & rtk_y.notna()
    if m_rtk.any():
        plt.plot(rtk_x[m_rtk].to_numpy(), rtk_y[m_rtk].to_numpy(), label="RTK Scene Trajectory", linewidth=2.0)

    m_uwb = uwb_x_aligned.notna() & uwb_y_aligned.notna()
    if m_uwb.any():
        plt.plot(
            uwb_x_aligned[m_uwb].to_numpy(),
            uwb_y_aligned[m_uwb].to_numpy(),
            label="Aligned UWB Scene Trajectory",
            linewidth=1.5
        )

    plt.xlabel("Scene X (m)")
    plt.ylabel("Scene Y (m)")
    plt.title("Aligned UWB vs RTK in Parking-Lot Scene Coordinates")
    plt.axis("equal")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=FIG_DPI)
    plt.close()

# =========================================================
# 11. Single-file processing
# =========================================================
def process_one_file(
    name: str,
    uwb_input_path: Path,
    rtk_txt_path: Path,
    log_path: Path,
    scene_ref: Dict[str, Any],
) -> Dict[str, Any]:
    file_stem = uwb_input_path.stem
    output_xlsx = OUTPUT_DIR / f"{file_stem}_uwb_rtk_aligned_result.xlsx"
    output_plot = OUTPUT_DIR / f"{file_stem}_uwb_rtk_aligned_trajectory.png"

    if log_path.exists():
        log_path.unlink()

    log_print(log_path, "=" * 90)
    log_print(log_path, f"开始处理: {name}")
    log_print(log_path, f"UWB: {uwb_input_path}")
    log_print(log_path, f"RTK: {rtk_txt_path}")
    log_print(log_path, f"输出 Excel: {output_xlsx}")
    log_print(log_path, f"输出轨迹图: {output_plot}")

    if not uwb_input_path.exists():
        raise FileNotFoundError(f"UWB 文件不存在: {uwb_input_path}")
    if not rtk_txt_path.exists():
        raise FileNotFoundError(f"RTK 文件不存在: {rtk_txt_path}")

    uwb_t, df_uwb = robust_load_uwb_csv_3ranges(uwb_input_path)
    df_ranges = df_uwb[["range0(m)", "range1(m)", "range2(m)"]].copy()

    log_print(log_path, f"[UWB] Samples = {len(df_ranges)}, estimated rate = {infer_rate(uwb_t):.3f} Hz")
    log_print(log_path, "[DEBUG] UWB dtypes:")
    append_log(log_path, str(df_uwb.dtypes))
    log_print(log_path, "[DEBUG] UWB head:")
    append_log(log_path, str(df_uwb.head(3)))

    anchors_xy = np.array([
        ANCHORS_2D[0],
        ANCHORS_2D[1],
        ANCHORS_2D[2],
    ], dtype=float)

    uwb_x_stage1, uwb_y_stage1, cross_residual, uwb_source_stage1 = solve_stage1_with_weak_prior(
        t=uwb_t,
        df_ranges=df_ranges,
        anchors_xy=anchors_xy,
    )

    uwb_x_filled, uwb_y_filled, uwb_source_filled = fill_short_gaps_linear(
        t=uwb_t,
        x=uwb_x_stage1,
        y=uwb_y_stage1,
        source=uwb_source_stage1,
        max_gap_points=MAX_GAP_FILL_POINTS,
    )

    uwb_x_smooth, uwb_y_smooth = smooth_trajectory_stage2(
        x=uwb_x_filled,
        y=uwb_y_filled,
        source=uwb_source_filled,
        window=SMOOTH_WIN,
    )

    uwb_x_flip, uwb_y_flip = simple_flip_uwb(uwb_x_smooth, uwb_y_smooth, flip_x=UWB_FLIP_X)

    # 统一停车场场景坐标系下的 RTK
    df_rtk = read_rtk_series_to_scene(rtk_txt_path, scene_ref=scene_ref)
    rtk_t = df_rtk["时间_s"].to_numpy(dtype=float)
    rtk_x_scene = df_rtk["RTK场景X_m"].to_numpy(dtype=float)
    rtk_y_scene = df_rtk["RTK场景Y_m"].to_numpy(dtype=float)

    log_print(log_path, f"[RTK] Samples = {len(df_rtk)}, estimated rate = {infer_rate(rtk_t):.3f} Hz")
    log_print(log_path, "[DEBUG] RTK dtypes:")
    append_log(log_path, str(df_rtk.dtypes))
    log_print(log_path, "[DEBUG] RTK head:")
    append_log(log_path, str(df_rtk.head(3)))

    # UWB 直接配准到统一场景坐标系下的 RTK
    best = search_best_time_shift_for_alignment(
        uwb_t=uwb_t,
        uwb_x=uwb_x_flip,
        uwb_y=uwb_y_flip,
        cross_residual=cross_residual,
        source=uwb_source_filled,
        rtk_t=rtk_t,
        rtk_x=rtk_x_scene,
        rtk_y=rtk_y_scene,
    )

    if best is None:
        raise ValueError(f"配准失败：{name} 没有找到可用时间偏移/空间配准结果。")

    log_print(
        log_path,
        f"[ALIGN] mode = {best['align_mode']}, points = {best['align_points']}, "
        f"time_shift = {best['time_shift_s']:.3f}, rmse = {best['rmse']:.4f}"
    )

    df_out = pd.DataFrame({
        "点序号": np.arange(1, len(uwb_t) + 1),
        "时间_s": pd.to_numeric(pd.Series(uwb_t), errors="coerce"),
        "rangetime(ms)": pd.to_numeric(df_uwb["rangetime(ms)"], errors="coerce") if "rangetime(ms)" in df_uwb.columns else np.nan,

        "range0(m)": pd.to_numeric(df_ranges["range0(m)"], errors="coerce"),
        "range1(m)": pd.to_numeric(df_ranges["range1(m)"], errors="coerce"),
        "range2(m)": pd.to_numeric(df_ranges["range2(m)"], errors="coerce"),

        "有效测距个数": np.isfinite(
            df_ranges[["range0(m)", "range1(m)", "range2(m)"]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
        ).sum(axis=1),

        "第一阶段UWB_X_m": pd.to_numeric(pd.Series(uwb_x_stage1), errors="coerce"),
        "第一阶段UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_stage1), errors="coerce"),

        "补全后UWB_X_m": pd.to_numeric(pd.Series(uwb_x_filled), errors="coerce"),
        "补全后UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_filled), errors="coerce"),

        "平滑后UWB_X_m": pd.to_numeric(pd.Series(uwb_x_smooth), errors="coerce"),
        "平滑后UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_smooth), errors="coerce"),

        "翻转后UWB_X_m": pd.to_numeric(pd.Series(uwb_x_flip), errors="coerce"),
        "翻转后UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_flip), errors="coerce"),

        "UWB位置来源": pd.Series(uwb_source_filled).astype(str),
        "三圆交叉残差_m": pd.to_numeric(pd.Series(cross_residual), errors="coerce"),

        "RTK插值场景X_m": pd.to_numeric(pd.Series(best["rtk_x_interp"]), errors="coerce"),
        "RTK插值场景Y_m": pd.to_numeric(pd.Series(best["rtk_y_interp"]), errors="coerce"),

        "UWB配准后场景X_m": pd.to_numeric(pd.Series(best["uwb_x_aligned"]), errors="coerce"),
        "UWB配准后场景Y_m": pd.to_numeric(pd.Series(best["uwb_y_aligned"]), errors="coerce"),

        "是否参与配准": pd.Series(best["align_mask"]).astype(int),
    })

    df_out["定位误差X_m"] = to_num_series(df_out["UWB配准后场景X_m"]) - to_num_series(df_out["RTK插值场景X_m"])
    df_out["定位误差Y_m"] = to_num_series(df_out["UWB配准后场景Y_m"]) - to_num_series(df_out["RTK插值场景Y_m"])
    df_out["定位误差_RTK_m"] = np.sqrt(df_out["定位误差X_m"] ** 2 + df_out["定位误差Y_m"] ** 2)

    err_s = to_num_series(df_out["定位误差_RTK_m"])
    res_s = to_num_series(df_out["三圆交叉残差_m"])

    valid_err = err_s.notna()
    valid_res = res_s.notna()

    source_counts = df_out["UWB位置来源"].astype(str).value_counts(dropna=False).to_dict()

    # 原始 RTK 场景坐标也一起写出，便于你检查停车场构建结果
    summary_row = {
        "数据组": name,
        "状态": "成功",
        "错误信息": "",
        "日志文件": str(log_path),
        "UWB输入文件": str(uwb_input_path),
        "RTK文件": str(rtk_txt_path),
        "总样本数": len(df_out),
        "可得到UWB位置点数": finite_count_2cols(df_out["UWB配准后场景X_m"], df_out["UWB配准后场景Y_m"]),
        "有RTK对应点数": finite_count_2cols(df_out["RTK插值场景X_m"], df_out["RTK插值场景Y_m"]),
        "可计算误差点数": int(valid_err.sum()),
        "最佳时间偏移_s": float(best["time_shift_s"]),
        "旋转角_deg": float(best["theta_deg"]),
        "平移X_m": float(best["t"][0]),
        "平移Y_m": float(best["t"][1]),
        "配准RMSE_m": float(best["rmse"]),
        "配准模式": str(best["align_mode"]),
        "配准点数": int(best["align_points"]),
        "三圆交叉残差均值_m": float(res_s[valid_res].mean()) if valid_res.any() else np.nan,
        "三圆交叉残差RMSE_m": float(np.sqrt((res_s[valid_res] ** 2).mean())) if valid_res.any() else np.nan,
        "定位误差_RTK均值_m": float(err_s[valid_err].mean()) if valid_err.any() else np.nan,
        "定位误差_RTK_RMSE_m": float(np.sqrt((err_s[valid_err] ** 2).mean())) if valid_err.any() else np.nan,
        "定位误差_RTK_95分位_m": float(err_s[valid_err].quantile(0.95)) if valid_err.any() else np.nan,
        "direct_3range点数": int(source_counts.get("direct_3range", 0)),
        "prior_2range点数": int(source_counts.get("prior_2range", 0)),
        "prior_1range点数": int(source_counts.get("prior_1range", 0)),
        "interp_gap点数": int(source_counts.get("interp_gap", 0)),
        "none_0range点数": int(source_counts.get("none_0range", 0)),
        "输出Excel": str(output_xlsx),
        "输出轨迹图": str(output_plot),
    }

    summary_df = pd.DataFrame([summary_row])

    with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
        df_out.to_excel(writer, sheet_name="逐点结果", index=False)
        summary_df.to_excel(writer, sheet_name="统计汇总", index=False)
        df_rtk.to_excel(writer, sheet_name="RTK原始转换结果", index=False)

    plot_aligned_trajectory(
        rtk_x=best["rtk_x_interp"],
        rtk_y=best["rtk_y_interp"],
        uwb_x_aligned=best["uwb_x_aligned"],
        uwb_y_aligned=best["uwb_y_aligned"],
        output_path=output_plot,
    )

    log_print(log_path, "Processing completed.")
    log_print(log_path, f"Best time shift (s): {best['time_shift_s']:.3f}")
    log_print(log_path, f"Alignment RMSE (m): {best['rmse']:.4f}")
    log_print(log_path, f"Excel saved to: {output_xlsx}")
    log_print(log_path, f"Plot saved to: {output_plot}")

    return summary_row

# =========================================================
# 12. Main
# =========================================================
def main():
    ensure_dir(OUTPUT_DIR)
    print(f"Output dir: {OUTPUT_DIR}")
    print(f"Batch summary: {OUTPUT_SUMMARY_XLSX}")

    # 先构建统一停车场场景坐标系
    scene_ref = build_global_parking_scene(DATA_PAIRS)

    scene_info_df = pd.DataFrame([{
        "全局原点纬度_deg": scene_ref["lat0"],
        "全局原点经度_deg": scene_ref["lon0"],
        "场景旋转角_deg": scene_ref["theta_scene_deg"],
        "场景平移X_m": scene_ref["shift_x"],
        "场景平移Y_m": scene_ref["shift_y"],
        "说明": "所有RTK已统一到该停车场场景坐标系，可直接用于后续分区与5x5网格划分",
    }])
    scene_info_df.to_excel(OUTPUT_SCENE_INFO_XLSX, index=False)

    all_summary_rows = []

    for item in DATA_PAIRS:
        log_path = OUTPUT_DIR / f"{item['name']}_run_log.txt"

        try:
            result = process_one_file(
                name=item["name"],
                uwb_input_path=item["uwb"],
                rtk_txt_path=item["rtk"],
                log_path=log_path,
                scene_ref=scene_ref,
            )
            all_summary_rows.append(result)

        except Exception as e:
            err_msg = str(e)
            tb = traceback.format_exc()

            print("=" * 90)
            print(f"[ERROR] 文件处理失败: {item['name']}")
            print(f"UWB: {item['uwb']}")
            print(f"RTK: {item['rtk']}")
            print(f"原因: {err_msg}")
            print(tb)

            append_log(log_path, "=" * 90)
            append_log(log_path, f"[ERROR] 文件处理失败: {item['name']}")
            append_log(log_path, f"UWB: {item['uwb']}")
            append_log(log_path, f"RTK: {item['rtk']}")
            append_log(log_path, f"原因: {err_msg}")
            append_log(log_path, tb)

            all_summary_rows.append({
                "数据组": item["name"],
                "状态": "失败",
                "错误信息": err_msg,
                "日志文件": str(log_path),
                "UWB输入文件": str(item["uwb"]),
                "RTK文件": str(item["rtk"]),
                "总样本数": np.nan,
                "可得到UWB位置点数": np.nan,
                "有RTK对应点数": np.nan,
                "可计算误差点数": np.nan,
                "最佳时间偏移_s": np.nan,
                "旋转角_deg": np.nan,
                "平移X_m": np.nan,
                "平移Y_m": np.nan,
                "配准RMSE_m": np.nan,
                "配准模式": "",
                "配准点数": np.nan,
                "三圆交叉残差均值_m": np.nan,
                "三圆交叉残差RMSE_m": np.nan,
                "定位误差_RTK均值_m": np.nan,
                "定位误差_RTK_RMSE_m": np.nan,
                "定位误差_RTK_95分位_m": np.nan,
                "direct_3range点数": np.nan,
                "prior_2range点数": np.nan,
                "prior_1range点数": np.nan,
                "interp_gap点数": np.nan,
                "none_0range点数": np.nan,
                "输出Excel": "",
                "输出轨迹图": "",
            })

    df_batch_summary = pd.DataFrame(all_summary_rows)
    df_batch_summary.to_excel(OUTPUT_SUMMARY_XLSX, index=False)

    print("=" * 90)
    print("全部处理完成。")
    print(f"批量汇总已保存到: {OUTPUT_SUMMARY_XLSX}")
    print(f"停车场统一场景参考已保存到: {OUTPUT_SCENE_INFO_XLSX}")

if __name__ == "__main__":
    main()


Output dir: D:\Desktop\missing_output\batch_output
Batch summary: D:\Desktop\missing_output\batch_output\uwb_rtk_batch_summary.xlsx
开始处理: 1试验
UWB: D:\Desktop\missing_output\1试验_missing.csv
RTK: D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\1.txt
输出 Excel: D:\Desktop\missing_output\batch_output\1试验_missing_uwb_rtk_aligned_result.xlsx
输出轨迹图: D:\Desktop\missing_output\batch_output\1试验_missing_uwb_rtk_aligned_trajectory.png
[UWB] Samples = 1435, estimated rate = 8.929 Hz
[DEBUG] UWB dtypes:
[DEBUG] UWB head:
[RTK] Samples = 2896, estimated rate = 10.000 Hz
[DEBUG] RTK dtypes:
[DEBUG] RTK head:
[ALIGN] mode = strict, points = 1385, time_shift = -10.300, rmse = 0.7836
Processing completed.
Best time shift (s): -10.300
Alignment RMSE (m): 0.7836
Excel saved to: D:\Desktop\missing_output\batch_output\1试验_missing_uwb_rtk_aligned_result.xlsx
Plot saved to: D:\Desktop\missing_output\batch_output\1试验_missing_uwb_rtk_aligned_trajectory.png
开始处理: 2圆形
UWB: D:\Desktop\missing_outpu

# 四锚点定位

In [30]:
import os
import numpy as np
import pandas as pd

# 输入文件
file_list = [
    r'D:\Desktop\paper2\数据\应用初始数据\1试验.csv',
    r'D:\Desktop\paper2\数据\应用初始数据\2圆形.csv',
    r'D:\Desktop\paper2\数据\应用初始数据\3方形.csv',
    r'D:\Desktop\paper2\数据\应用初始数据\4凹凸.csv'
]

# 需要提取并处理的列：4个锚点
extract_cols = [
    'rangetime(ms)',
    'range0(m)',
    'range1(m)',
    'range2(m)',
    'range3(m)'
]

# 对4个锚点列添加缺失
target_cols = [
    'range0(m)',
    'range1(m)',
    'range2(m)',
    'range3(m)'
]

# 每个文件中4个锚点各自的缺失比例
missing_rates = np.array([
    [0.10, 0.10, 0.10, 0.10],   # 1试验
    [0.05, 0.10, 0.05, 0.05],   # 2圆形
    [0.05, 0.05, 0.10, 0.05],   # 3方形
    [0.10, 0.10, 0.05, 0.10]    # 4凹凸
])

# 固定随机种子
np.random.seed(36)

# 输出文件夹
output_folder = r'D:\Desktop\missing_output'
os.makedirs(output_folder, exist_ok=True)

for f, file_path in enumerate(file_list):

    # 读取 CSV
    df = pd.read_csv(file_path, encoding='utf-8-sig')

    # 只提取需要的列
    df = df[extract_cols].copy()

    n = len(df)

    # 对4个锚点分别制造缺失
    for j, col in enumerate(target_cols):

        rate = missing_rates[f, j]
        n_missing = round(n * rate)

        # 当前列单独随机抽取缺失行
        idx = np.random.permutation(n)[:n_missing]

        # 置为缺失
        df.loc[idx, col] = np.nan

    # 输出文件名
    name = os.path.splitext(os.path.basename(file_path))[0]
    out_file = os.path.join(output_folder, f'{name}_missing.csv')

    # 保存为 CSV
    df.to_csv(out_file, index=False, encoding='utf-8-sig')

    print(f'已保存：{out_file}')


已保存：D:\Desktop\missing_output\1试验_missing.csv
已保存：D:\Desktop\missing_output\2圆形_missing.csv
已保存：D:\Desktop\missing_output\3方形_missing.csv
已保存：D:\Desktop\missing_output\4凹凸_missing.csv


In [31]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import math
import traceback
from pathlib import Path
from typing import Tuple, Optional, Dict, Any, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# 0. Matplotlib config
# =========================================================
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# =========================================================
# 1. Paths and parameters
# =========================================================
DATA_PAIRS = [
    {
        "name": "1试验",
        "uwb": Path(r"D:\Desktop\missing_output\1试验_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\1.txt"),
    },
    {
        "name": "2圆形",
        "uwb": Path(r"D:\Desktop\missing_output\2圆形_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\2.txt"),
    },
    {
        "name": "3方形",
        "uwb": Path(r"D:\Desktop\missing_output\3方形_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\3.txt"),
    },
    {
        "name": "4凹凸",
        "uwb": Path(r"D:\Desktop\missing_output\4凹凸_missing.csv"),
        "rtk": Path(r"D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\4.txt"),
    },
]

OUTPUT_DIR = Path(r"D:\Desktop\missing_output\batch_output")
OUTPUT_SUMMARY_XLSX = OUTPUT_DIR / "uwb_rtk_batch_summary.xlsx"
OUTPUT_SCENE_INFO_XLSX = OUTPUT_DIR / "parking_scene_reference.xlsx"

ANCHORS_2D = {
    0: (0.00, 0.00),
    1: (14.95, 0.00),
    2: (18.51, 33.15),
    3: (2.30, 24.79),
}

ANCHOR_IDS = [0, 1, 2, 3]
RANGE_COLS = ["range0(m)", "range1(m)", "range2(m)", "range3(m)"]

UWB_FLIP_X = True

TIME_SHIFT_CANDIDATES_COARSE = np.arange(-10.0, 10.01, 0.2)
TIME_SHIFT_FINE_STEP = 0.02
TIME_SHIFT_FINE_HALF_WIDTH = 0.3

FIG_DPI = 220

MAX_SPEED_MPS = 3.0
PREDICTION_BLEND = 0.80
MIN_DT_FOR_VEL = 1e-6

SMOOTH_WIN = 5
SMOOTH_ENABLE = True

MAX_CROSS_RESIDUAL_FOR_ALIGNMENT = 1.0
MIN_ALIGNMENT_POINTS = 8
MAX_CROSS_RESIDUAL_FOR_ALIGNMENT_RELAXED = 2.5
MIN_ALIGNMENT_POINTS_RELAXED = 5

MAX_GAP_FILL_POINTS = 8

# =========================================================
# 2. Logging helpers
# =========================================================
def ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def append_log(log_path: Path, text: str) -> None:
    ensure_parent_dir(log_path)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(text.rstrip() + "\n")


def log_print(log_path: Path, text: str) -> None:
    print(text)
    append_log(log_path, text)


def to_num_series(s):
    return pd.to_numeric(s, errors="coerce")


def infer_rate(t: np.ndarray) -> float:
    t = np.asarray(t, dtype=float)
    if len(t) < 3:
        return float("nan")
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        return float("nan")
    return 1.0 / float(np.median(dt))


def finite_count_2cols(a, b) -> int:
    a1 = pd.to_numeric(pd.Series(a), errors="coerce")
    b1 = pd.to_numeric(pd.Series(b), errors="coerce")
    return int((a1.notna() & b1.notna()).sum())


def interp1_nan_safe(t_ref, v_ref, t_query):
    t_ref = np.asarray(t_ref, dtype=float)
    v_ref = np.asarray(v_ref, dtype=float)
    t_query = np.asarray(t_query, dtype=float)

    m = np.isfinite(t_ref) & np.isfinite(v_ref)
    if np.count_nonzero(m) < 2:
        return np.full_like(t_query, np.nan, dtype=float)

    t0 = t_ref[m]
    v0 = v_ref[m]
    order = np.argsort(t0)
    t0 = t0[order]
    v0 = v0[order]

    out = np.interp(t_query, t0, v0)
    out[(t_query < t0[0]) | (t_query > t0[-1])] = np.nan
    return out


def interp2_at_times(t_ref, x_ref, y_ref, t_query):
    xi = interp1_nan_safe(t_ref, x_ref, t_query)
    yi = interp1_nan_safe(t_ref, y_ref, t_query)
    return xi, yi


def moving_average_1d_preserve_nan(x: np.ndarray, window: int = 5) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if window < 2:
        return x.copy()
    if window % 2 == 0:
        window += 1

    y = x.copy()
    pad = window // 2
    for i in range(len(x)):
        lo = max(0, i - pad)
        hi = min(len(x), i + pad + 1)
        seg = x[lo:hi]
        if np.count_nonzero(np.isfinite(seg)) >= 1:
            y[i] = np.nanmean(seg)
        else:
            y[i] = np.nan
    return y


def smooth_xy(x: np.ndarray, y: np.ndarray, window: int = 5) -> Tuple[np.ndarray, np.ndarray]:
    return moving_average_1d_preserve_nan(x, window), moving_average_1d_preserve_nan(y, window)


def calc_rmse(x1, y1, x2, y2):
    x1 = np.asarray(x1, dtype=float)
    y1 = np.asarray(y1, dtype=float)
    x2 = np.asarray(x2, dtype=float)
    y2 = np.asarray(y2, dtype=float)

    m = np.isfinite(x1) & np.isfinite(y1) & np.isfinite(x2) & np.isfinite(y2)
    if np.count_nonzero(m) < 3:
        return np.inf
    err = np.sqrt((x1[m] - x2[m]) ** 2 + (y1[m] - y2[m]) ** 2)
    return float(np.sqrt(np.mean(err ** 2)))


def is_valid_number(v) -> bool:
    try:
        return bool(np.isfinite(float(v)))
    except Exception:
        return False


def is_valid_xy(x, y) -> bool:
    return is_valid_number(x) and is_valid_number(y)


# =========================================================
# 3. Load UWB CSV/XLSX
# =========================================================
def robust_load_uwb_csv_4ranges(file_path: Path) -> Tuple[np.ndarray, pd.DataFrame]:
    if not file_path.exists():
        raise FileNotFoundError(f"UWB input file not found: {file_path}")

    if file_path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(
            file_path,
            engine="python",
            encoding="utf-8",
            na_values=["null", "NULL", "", "NaN", "nan"],
            keep_default_na=True,
        )

    df.columns = [str(c).strip() for c in df.columns]

    time_col = None
    for c in ["rangetime(ms)", "rangetime", "time", "时间_s", "time_s", "t"]:
        if c in df.columns:
            time_col = c
            break
    if time_col is None:
        raise ValueError("未找到时间列，请至少包含 rangetime(ms) 或 时间_s。")

    missing = [c for c in RANGE_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"文件缺少必要列: {missing}")

    numeric_cols = [time_col] + RANGE_COLS
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    t_raw = df[time_col].to_numpy(dtype=float)
    if np.isfinite(t_raw).sum() < 2:
        raise ValueError(f"{time_col} 列有效数据不足。")

    first_valid = t_raw[np.isfinite(t_raw)][0]
    if time_col == "rangetime(ms)" or "ms" in time_col.lower():
        t = (t_raw - first_valid) / 1000.0
    else:
        t = t_raw - first_valid

    idx = np.arange(len(t), dtype=float)
    m = np.isfinite(t)
    if np.count_nonzero(m) >= 2 and np.count_nonzero(~m) > 0:
        t[~m] = np.interp(idx[~m], idx[m], t[m])

    return t, df.copy()


# =========================================================
# 4. RTK parsing and global parking-lot scene system
# =========================================================
def nmea_degmin_to_deg(dm: str, hemi: str) -> float:
    if not dm:
        return float("nan")
    try:
        dm = float(dm)
    except Exception:
        return float("nan")

    deg = int(dm // 100)
    minutes = dm - deg * 100
    val = deg + minutes / 60.0
    if hemi in ("S", "W"):
        val = -val
    return val


def parse_gga_time_to_sec(tfield: str) -> Optional[float]:
    if not tfield:
        return None
    try:
        hh = int(tfield[0:2])
        mm = int(tfield[2:4])
        ss = float(tfield[4:])
        return hh * 3600 + mm * 60 + ss
    except Exception:
        return None


def make_time_monotonic(t_raw: np.ndarray) -> np.ndarray:
    t_raw = np.asarray(t_raw, dtype=float)
    if t_raw.size == 0:
        return t_raw.copy()

    t = np.array(t_raw, dtype=float)
    day_offset = 0.0
    prev = t[0]

    for i in range(1, len(t)):
        cur = t[i]
        while cur + day_offset < prev - 1.0:
            day_offset += 86400.0
        adj = cur + day_offset
        if adj < prev:
            adj = prev
        t[i] = adj
        prev = adj

    return t


def geodetic_to_local_xy(lat: float, lon: float, lat0: float, lon0: float) -> Tuple[float, float]:
    R = 6378137.0
    lat0r = math.radians(lat0)
    x = math.radians(lon - lon0) * math.cos(lat0r) * R
    y = math.radians(lat - lat0) * R
    return x, y


def parse_rtk_gga_raw(rtk_log_path: Path) -> pd.DataFrame:
    if not rtk_log_path.exists():
        raise FileNotFoundError(f"RTK file not found: {rtk_log_path}")

    times_raw = []
    lats_raw = []
    lons_raw = []

    with open(rtk_log_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not (line.startswith("$") and "GGA" in line):
                continue

            parts = line.split(",")
            if len(parts) < 7:
                continue

            fix = parts[6]
            if fix not in ("4", "5", "2", "1"):
                continue

            tsec = parse_gga_time_to_sec(parts[1] if len(parts) > 1 else "")
            lat = nmea_degmin_to_deg(parts[2], parts[3] if len(parts) > 3 else "")
            lon = nmea_degmin_to_deg(parts[4], parts[5] if len(parts) > 5 else "")

            if (tsec is None) or (not np.isfinite(lat)) or (not np.isfinite(lon)):
                continue

            times_raw.append(float(tsec))
            lats_raw.append(float(lat))
            lons_raw.append(float(lon))

    if len(times_raw) < 2:
        raise ValueError(f"RTK 文件有效 GGA 点不足: {rtk_log_path}")

    t = make_time_monotonic(np.array(times_raw, float))
    t = t - t[0]

    return pd.DataFrame({
        "时间_s": t,
        "纬度_deg": np.array(lats_raw, float),
        "经度_deg": np.array(lons_raw, float),
    })


def estimate_scene_rotation_from_points(x_all: np.ndarray, y_all: np.ndarray) -> float:
    pts = np.column_stack([x_all, y_all]).astype(float)
    m = np.isfinite(pts).all(axis=1)
    pts = pts[m]
    if len(pts) < 3:
        return 0.0

    pts0 = pts - pts.mean(axis=0, keepdims=True)
    cov = np.cov(pts0.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    v = eigvecs[:, np.argmax(eigvals)]

    theta = math.atan2(v[1], v[0])
    rot = -theta

    return float(rot)


def rotate_xy(x: np.ndarray, y: np.ndarray, theta_rad: float) -> Tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    c = math.cos(theta_rad)
    s = math.sin(theta_rad)
    xr = c * x - s * y
    yr = s * x + c * y
    return xr, yr


def build_global_parking_scene(data_pairs: List[Dict[str, Path]]) -> Dict[str, Any]:
    all_lat = []
    all_lon = []
    raw_dfs = {}

    for item in data_pairs:
        df_raw = parse_rtk_gga_raw(item["rtk"])
        raw_dfs[item["name"]] = df_raw

        all_lat.extend(df_raw["纬度_deg"].tolist())
        all_lon.extend(df_raw["经度_deg"].tolist())

    if len(all_lat) == 0:
        raise ValueError("全部 RTK 文件都没有有效经纬度，无法构建统一停车场场景坐标系。")

    lat0 = float(np.median(np.asarray(all_lat, dtype=float)))
    lon0 = float(np.median(np.asarray(all_lon, dtype=float)))

    all_x_enu = []
    all_y_enu = []

    for _, df_raw in raw_dfs.items():
        lat_arr = df_raw["纬度_deg"].to_numpy(dtype=float)
        lon_arr = df_raw["经度_deg"].to_numpy(dtype=float)
        xe = np.empty_like(lat_arr)
        yn = np.empty_like(lat_arr)
        for i in range(len(lat_arr)):
            xe[i], yn[i] = geodetic_to_local_xy(lat_arr[i], lon_arr[i], lat0, lon0)
        all_x_enu.append(xe)
        all_y_enu.append(yn)

    x_all = np.concatenate(all_x_enu)
    y_all = np.concatenate(all_y_enu)

    theta_scene = estimate_scene_rotation_from_points(x_all, y_all)

    x_scene_all, y_scene_all = rotate_xy(x_all, y_all, theta_scene)

    scene_min_x = np.nanmin(x_scene_all)
    scene_min_y = np.nanmin(y_scene_all)
    tx = -scene_min_x
    ty = -scene_min_y

    return {
        "lat0": lat0,
        "lon0": lon0,
        "theta_scene_rad": theta_scene,
        "theta_scene_deg": math.degrees(theta_scene),
        "shift_x": tx,
        "shift_y": ty,
        "raw_dfs": raw_dfs,
    }


def read_rtk_series_to_scene(rtk_log_path: Path, scene_ref: Dict[str, Any]) -> pd.DataFrame:
    df_raw = parse_rtk_gga_raw(rtk_log_path)

    lat0 = scene_ref["lat0"]
    lon0 = scene_ref["lon0"]
    theta_scene = scene_ref["theta_scene_rad"]
    shift_x = scene_ref["shift_x"]
    shift_y = scene_ref["shift_y"]

    lat_arr = df_raw["纬度_deg"].to_numpy(dtype=float)
    lon_arr = df_raw["经度_deg"].to_numpy(dtype=float)

    E = np.empty_like(lat_arr)
    N = np.empty_like(lat_arr)
    for i in range(len(lat_arr)):
        E[i], N[i] = geodetic_to_local_xy(lat_arr[i], lon_arr[i], lat0, lon0)

    X_scene, Y_scene = rotate_xy(E, N, theta_scene)
    X_scene = X_scene + shift_x
    Y_scene = Y_scene + shift_y

    return pd.DataFrame({
        "时间_s": df_raw["时间_s"].to_numpy(dtype=float),
        "RTK_ENU_X_m": E,
        "RTK_ENU_Y_m": N,
        "RTK场景X_m": X_scene,
        "RTK场景Y_m": Y_scene,
        "纬度_deg": lat_arr,
        "经度_deg": lon_arr,
    })


# =========================================================
# 5. Geometry helpers
# =========================================================
def trilaterate_2d_three_anchors(anchors_xy: np.ndarray, d: np.ndarray) -> Tuple[float, float]:
    anchors_xy = np.asarray(anchors_xy, dtype=float)
    d = np.asarray(d, dtype=float)

    if anchors_xy.shape[0] != 3 or len(d) != 3:
        return np.nan, np.nan
    if not (np.isfinite(anchors_xy).all() and np.isfinite(d).all()):
        return np.nan, np.nan

    (x1, y1), (x2, y2), (x3, y3) = anchors_xy
    d1, d2, d3 = d

    A = np.array([
        [2 * (x2 - x1), 2 * (y2 - y1)],
        [2 * (x3 - x1), 2 * (y3 - y1)],
    ], dtype=float)

    b = np.array([
        (x2**2 - x1**2) + (y2**2 - y1**2) + (d1**2 - d2**2),
        (x3**2 - x1**2) + (y3**2 - y1**2) + (d1**2 - d3**2),
    ], dtype=float)

    if abs(np.linalg.det(A)) < 1e-12:
        return np.nan, np.nan

    try:
        sol = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        return np.nan, np.nan
    return float(sol[0]), float(sol[1])


def trilaterate_2d_multi_anchors(anchors_xy: np.ndarray, d: np.ndarray) -> Tuple[float, float]:
    anchors_xy = np.asarray(anchors_xy, dtype=float)
    d = np.asarray(d, dtype=float)

    valid = np.isfinite(d) & np.isfinite(anchors_xy).all(axis=1)
    anchors_xy = anchors_xy[valid]
    d = d[valid]

    if anchors_xy.shape[0] < 3:
        return np.nan, np.nan

    if anchors_xy.shape[0] == 3:
        return trilaterate_2d_three_anchors(anchors_xy, d)

    x1, y1 = anchors_xy[0]
    d1 = d[0]

    A = []
    b = []
    for i in range(1, len(d)):
        xi, yi = anchors_xy[i]
        di = d[i]
        A.append([2 * (xi - x1), 2 * (yi - y1)])
        b.append((xi**2 - x1**2) + (yi**2 - y1**2) + (d1**2 - di**2))

    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)

    try:
        sol, *_ = np.linalg.lstsq(A, b, rcond=None)
        return float(sol[0]), float(sol[1])
    except Exception:
        return np.nan, np.nan


def circle_circle_intersections(
    c0: Tuple[float, float],
    r0: float,
    c1: Tuple[float, float],
    r1: float
) -> List[Tuple[float, float]]:
    x0, y0 = c0
    x1, y1 = c1

    dx = x1 - x0
    dy = y1 - y0
    d = math.hypot(dx, dy)

    if not np.isfinite(d) or d < 1e-12:
        return []
    if d > r0 + r1:
        return []
    if d < abs(r0 - r1):
        return []
    if d == 0 and abs(r0 - r1) < 1e-12:
        return []

    a = (r0**2 - r1**2 + d**2) / (2 * d)
    h2 = r0**2 - a**2
    if h2 < -1e-10:
        return []
    h = math.sqrt(max(h2, 0.0))

    xm = x0 + a * dx / d
    ym = y0 + a * dy / d

    rx = -dy * (h / d)
    ry = dx * (h / d)

    p1 = (xm + rx, ym + ry)
    p2 = (xm - rx, ym - ry)

    if math.hypot(p1[0] - p2[0], p1[1] - p2[1]) < 1e-10:
        return [p1]
    return [p1, p2]


def project_point_to_circle(
    center: Tuple[float, float],
    radius: float,
    point: Tuple[float, float]
) -> Tuple[float, float]:
    cx, cy = center
    px, py = point

    vx = px - cx
    vy = py - cy
    norm = math.hypot(vx, vy)

    if norm < 1e-12:
        return cx + radius, cy
    scale = radius / norm
    return cx + vx * scale, cy + vy * scale


def compute_cross_residual_single(x: float, y: float, d: np.ndarray, anchors_xy: np.ndarray) -> float:
    d = np.asarray(d, dtype=float)
    anchors_xy = np.asarray(anchors_xy, dtype=float)

    if not (np.isfinite(x) and np.isfinite(y)):
        return np.nan
    if len(d) < 3 or anchors_xy.shape[0] != len(d):
        return np.nan
    if not (np.all(np.isfinite(d)) and np.isfinite(anchors_xy).all()):
        return np.nan

    errs = []
    for i in range(len(d)):
        ax, ay = anchors_xy[i]
        pred = math.sqrt((x - ax) ** 2 + (y - ay) ** 2)
        errs.append(pred - d[i])

    errs = np.asarray(errs, dtype=float)
    return float(np.sqrt(np.mean(errs ** 2)))


# =========================================================
# 6. Weak-prior stage 1 trajectory reconstruction
# =========================================================
def build_prediction(
    t: np.ndarray,
    x_prev: Optional[float],
    y_prev: Optional[float],
    x_prev2: Optional[float],
    y_prev2: Optional[float],
    i: int,
) -> Tuple[float, float]:
    if not is_valid_xy(x_prev, y_prev):
        return np.nan, np.nan

    if not is_valid_xy(x_prev2, y_prev2) or i < 2:
        return float(x_prev), float(y_prev)

    dt1 = max(t[i - 1] - t[i - 2], MIN_DT_FOR_VEL)
    dt2 = max(t[i] - t[i - 1], MIN_DT_FOR_VEL)

    vx = (float(x_prev) - float(x_prev2)) / dt1
    vy = (float(y_prev) - float(y_prev2)) / dt1

    x_pred = float(x_prev) + PREDICTION_BLEND * vx * dt2
    y_pred = float(y_prev) + PREDICTION_BLEND * vy * dt2
    return x_pred, y_pred


def limit_jump(
    x_candidate: float,
    y_candidate: float,
    x_prev: Optional[float],
    y_prev: Optional[float],
    dt: float,
    max_speed_mps: float = MAX_SPEED_MPS
) -> Tuple[float, float]:
    if not is_valid_xy(x_candidate, y_candidate):
        return np.nan, np.nan

    if not is_valid_xy(x_prev, y_prev):
        return float(x_candidate), float(y_candidate)

    max_dist = max_speed_mps * max(dt, MIN_DT_FOR_VEL)
    dx = float(x_candidate) - float(x_prev)
    dy = float(y_candidate) - float(y_prev)
    dist = math.hypot(dx, dy)

    if dist <= max_dist or dist < 1e-12:
        return float(x_candidate), float(y_candidate)

    scale = max_dist / dist
    return float(x_prev) + dx * scale, float(y_prev) + dy * scale


def solve_stage1_with_weak_prior(
    t: np.ndarray,
    df_ranges: pd.DataFrame,
    anchors_xy: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    vals = df_ranges[RANGE_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    n = len(vals)

    x = np.full(n, np.nan, dtype=float)
    y = np.full(n, np.nan, dtype=float)
    cross_residual = np.full(n, np.nan, dtype=float)
    source = np.full(n, "", dtype=object)

    prev_valid_indices: List[int] = []

    for i in range(n):
        row = vals[i]
        valid_idx = np.where(np.isfinite(row))[0]
        n_valid = len(valid_idx)

        x_prev = y_prev = x_prev2 = y_prev2 = None
        if len(prev_valid_indices) >= 1:
            j1 = prev_valid_indices[-1]
            x_prev, y_prev = float(x[j1]), float(y[j1])
        if len(prev_valid_indices) >= 2:
            j2 = prev_valid_indices[-2]
            x_prev2, y_prev2 = float(x[j2]), float(y[j2])

        x_pred, y_pred = build_prediction(t, x_prev, y_prev, x_prev2, y_prev2, i)

        if n_valid >= 3:
            anchors_valid = anchors_xy[valid_idx]
            ranges_valid = row[valid_idx]

            xi, yi = trilaterate_2d_multi_anchors(anchors_valid, ranges_valid)
            if is_valid_xy(xi, yi):
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)

                x[i], y[i] = xi, yi
                cross_residual[i] = compute_cross_residual_single(
                    xi,
                    yi,
                    ranges_valid,
                    anchors_valid,
                )
                source[i] = f"direct_{n_valid}range"
                prev_valid_indices.append(i)
            else:
                source[i] = f"bad_{n_valid}range"
            continue

        if n_valid == 2:
            a0, a1 = valid_idx.tolist()
            p0 = tuple(anchors_xy[a0])
            p1 = tuple(anchors_xy[a1])
            r0 = float(row[a0])
            r1 = float(row[a1])

            cands = circle_circle_intersections(p0, r0, p1, r1)
            if len(cands) > 0:
                if is_valid_xy(x_pred, y_pred):
                    dists = [math.hypot(cx - x_pred, cy - y_pred) for cx, cy in cands]
                    best_idx = int(np.argmin(dists))
                elif is_valid_xy(x_prev, y_prev):
                    dists = [math.hypot(cx - x_prev, cy - y_prev) for cx, cy in cands]
                    best_idx = int(np.argmin(dists))
                else:
                    best_idx = 0

                xi, yi = cands[best_idx]
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)

                x[i], y[i] = xi, yi
                source[i] = "prior_2range"
                prev_valid_indices.append(i)
            else:
                source[i] = "bad_2range"
            continue

        if n_valid == 1:
            a0 = int(valid_idx[0])
            p0 = tuple(anchors_xy[a0])
            r0 = float(row[a0])

            if is_valid_xy(x_pred, y_pred):
                xi, yi = project_point_to_circle(p0, r0, (x_pred, y_pred))
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)

                x[i], y[i] = xi, yi
                source[i] = "prior_1range"
                prev_valid_indices.append(i)
            elif is_valid_xy(x_prev, y_prev):
                xi, yi = project_point_to_circle(p0, r0, (x_prev, y_prev))
                dt = (t[i] - t[prev_valid_indices[-1]]) if len(prev_valid_indices) >= 1 else 0.0
                xi, yi = limit_jump(xi, yi, x_prev, y_prev, dt)

                x[i], y[i] = xi, yi
                source[i] = "prior_1range"
                prev_valid_indices.append(i)
            else:
                source[i] = "none_1range_no_prior"
            continue

        source[i] = "none_0range"

    return x, y, cross_residual, source


# =========================================================
# 7. Fill short gaps for 0-range cases
# =========================================================
def fill_short_gaps_linear(
    t: np.ndarray,
    x: np.ndarray,
    y: np.ndarray,
    source: np.ndarray,
    max_gap_points: int = 8
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    x2 = np.asarray(x, dtype=float).copy()
    y2 = np.asarray(y, dtype=float).copy()
    source2 = source.copy()

    n = len(x2)
    i = 0
    while i < n:
        if np.isfinite(x2[i]) and np.isfinite(y2[i]):
            i += 1
            continue

        j = i
        while j < n and not (np.isfinite(x2[j]) and np.isfinite(y2[j])):
            j += 1

        gap_len = j - i
        left = i - 1
        right = j

        can_fill = (
            gap_len <= max_gap_points and
            left >= 0 and right < n and
            np.isfinite(x2[left]) and np.isfinite(y2[left]) and
            np.isfinite(x2[right]) and np.isfinite(y2[right])
        )

        if can_fill:
            for k in range(i, j):
                alpha = (t[k] - t[left]) / max(t[right] - t[left], MIN_DT_FOR_VEL)
                x2[k] = (1 - alpha) * x2[left] + alpha * x2[right]
                y2[k] = (1 - alpha) * y2[left] + alpha * y2[right]
                if (
                    source2[k] in ["none_0range", "none_1range_no_prior", "bad_2range", "bad_3range", "bad_4range"]
                    or str(source2[k]).startswith("bad_")
                ):
                    source2[k] = "interp_gap"
        i = j

    return x2, y2, source2


# =========================================================
# 8. Stage 2 smoothing
# =========================================================
def smooth_trajectory_stage2(
    x: np.ndarray,
    y: np.ndarray,
    source: np.ndarray,
    window: int = 5
) -> Tuple[np.ndarray, np.ndarray]:
    if not SMOOTH_ENABLE:
        return np.asarray(x, dtype=float).copy(), np.asarray(y, dtype=float).copy()

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x_s, y_s = smooth_xy(x, y, window=window)

    strong_mask = np.isin(source, ["direct_4range", "direct_3range", "prior_2range"])
    x_s[strong_mask] = x[strong_mask]
    y_s[strong_mask] = y[strong_mask]

    return x_s, y_s


# =========================================================
# 9. Alignment
# =========================================================
def simple_flip_uwb(x: np.ndarray, y: np.ndarray, flip_x=True):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if flip_x:
        return -x, y
    return x.copy(), y.copy()


def kabsch_2d(P, Q):
    P = np.asarray(P, dtype=float)
    Q = np.asarray(Q, dtype=float)

    Pc = P - P.mean(axis=0, keepdims=True)
    Qc = Q - Q.mean(axis=0, keepdims=True)

    H = Pc.T @ Qc
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    t = Q.mean(axis=0) - (R @ P.mean(axis=0))
    theta_deg = math.degrees(math.atan2(R[1, 0], R[0, 0]))
    return R, t, theta_deg


def apply_rigid_transform(x, y, R, t):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    XY = np.column_stack([x, y]).astype(float)
    XY2 = (R @ XY.T).T + t
    return XY2[:, 0], XY2[:, 1]


def build_alignment_mask_strict(uwb_x, uwb_y, cross_residual, source, rtk_xi, rtk_yi):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    cross_residual = np.asarray(cross_residual, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    strong_source_mask = np.isin(source, ["direct_4range", "direct_3range", "prior_2range"])
    residual_ok = np.isfinite(cross_residual) & (cross_residual <= MAX_CROSS_RESIDUAL_FOR_ALIGNMENT)
    prior2_ok = (source == "prior_2range")

    return (
        np.isfinite(uwb_x) &
        np.isfinite(uwb_y) &
        np.isfinite(rtk_xi) &
        np.isfinite(rtk_yi) &
        strong_source_mask &
        (residual_ok | prior2_ok)
    )


def build_alignment_mask_relaxed(uwb_x, uwb_y, cross_residual, source, rtk_xi, rtk_yi):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    cross_residual = np.asarray(cross_residual, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    source_ok = np.isin(source, ["direct_4range", "direct_3range", "prior_2range", "prior_1range", "interp_gap"])
    residual_ok = (~np.isfinite(cross_residual)) | (cross_residual <= MAX_CROSS_RESIDUAL_FOR_ALIGNMENT_RELAXED)

    return (
        np.isfinite(uwb_x) &
        np.isfinite(uwb_y) &
        np.isfinite(rtk_xi) &
        np.isfinite(rtk_yi) &
        source_ok &
        residual_ok
    )


def build_alignment_mask_fallback_allfinite(uwb_x, uwb_y, rtk_xi, rtk_yi):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    return (
        np.isfinite(uwb_x) &
        np.isfinite(uwb_y) &
        np.isfinite(rtk_xi) &
        np.isfinite(rtk_yi)
    )


def evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, mask):
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    rtk_xi = np.asarray(rtk_xi, dtype=float)
    rtk_yi = np.asarray(rtk_yi, dtype=float)

    if np.count_nonzero(mask) < 3:
        return None

    P = np.column_stack([uwb_x[mask], uwb_y[mask]])
    Q = np.column_stack([rtk_xi[mask], rtk_yi[mask]])

    R, t, theta_deg = kabsch_2d(P, Q)
    uwb_x2, uwb_y2 = apply_rigid_transform(uwb_x, uwb_y, R, t)
    score = calc_rmse(uwb_x2[mask], uwb_y2[mask], rtk_xi[mask], rtk_yi[mask])

    return {
        "R": R,
        "t": t,
        "theta_deg": float(theta_deg),
        "rmse": float(score),
        "uwb_x_aligned": uwb_x2,
        "uwb_y_aligned": uwb_y2,
    }


def search_best_time_shift_for_alignment(
    uwb_t, uwb_x, uwb_y, cross_residual, source, rtk_t, rtk_x, rtk_y
):
    uwb_t = np.asarray(uwb_t, dtype=float)
    uwb_x = np.asarray(uwb_x, dtype=float)
    uwb_y = np.asarray(uwb_y, dtype=float)
    cross_residual = np.asarray(cross_residual, dtype=float)
    rtk_t = np.asarray(rtk_t, dtype=float)
    rtk_x = np.asarray(rtk_x, dtype=float)
    rtk_y = np.asarray(rtk_y, dtype=float)

    best = None
    best_score = np.inf

    def _search(time_shift_candidates):
        nonlocal best, best_score

        for dt in time_shift_candidates:
            rtk_xi, rtk_yi = interp2_at_times(rtk_t + dt, rtk_x, rtk_y, uwb_t)

            candidate = None
            mode = None
            align_mask = None

            m_strict = build_alignment_mask_strict(
                uwb_x=uwb_x,
                uwb_y=uwb_y,
                cross_residual=cross_residual,
                source=source,
                rtk_xi=rtk_xi,
                rtk_yi=rtk_yi,
            )
            if np.count_nonzero(m_strict) >= MIN_ALIGNMENT_POINTS:
                out = evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, m_strict)
                if out is not None:
                    candidate = out
                    mode = "strict"
                    align_mask = m_strict

            if candidate is None:
                m_relaxed = build_alignment_mask_relaxed(
                    uwb_x=uwb_x,
                    uwb_y=uwb_y,
                    cross_residual=cross_residual,
                    source=source,
                    rtk_xi=rtk_xi,
                    rtk_yi=rtk_yi,
                )
                if np.count_nonzero(m_relaxed) >= MIN_ALIGNMENT_POINTS_RELAXED:
                    out = evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, m_relaxed)
                    if out is not None:
                        candidate = out
                        mode = "relaxed"
                        align_mask = m_relaxed

            if candidate is None:
                m_all = build_alignment_mask_fallback_allfinite(uwb_x, uwb_y, rtk_xi, rtk_yi)
                if np.count_nonzero(m_all) >= 3:
                    out = evaluate_alignment_with_mask(uwb_x, uwb_y, rtk_xi, rtk_yi, m_all)
                    if out is not None:
                        candidate = out
                        mode = "fallback_allfinite"
                        align_mask = m_all

            if candidate is None:
                continue

            score = candidate["rmse"]
            if score < best_score:
                best_score = score
                best = {
                    "time_shift_s": float(dt),
                    "R": candidate["R"],
                    "t": candidate["t"],
                    "theta_deg": candidate["theta_deg"],
                    "rmse": candidate["rmse"],
                    "rtk_x_interp": rtk_xi,
                    "rtk_y_interp": rtk_yi,
                    "uwb_x_aligned": candidate["uwb_x_aligned"],
                    "uwb_y_aligned": candidate["uwb_y_aligned"],
                    "align_mask": align_mask,
                    "align_mode": mode,
                    "align_points": int(np.count_nonzero(align_mask)),
                }

    _search(TIME_SHIFT_CANDIDATES_COARSE)
    if best is None:
        return None

    c = best["time_shift_s"]
    fine_candidates = np.arange(
        c - TIME_SHIFT_FINE_HALF_WIDTH,
        c + TIME_SHIFT_FINE_HALF_WIDTH + TIME_SHIFT_FINE_STEP,
        TIME_SHIFT_FINE_STEP
    )
    _search(fine_candidates)
    return best


# =========================================================
# 10. Plot
# =========================================================
def plot_aligned_trajectory(rtk_x, rtk_y, uwb_x_aligned, uwb_y_aligned, output_path: Path):
    ensure_parent_dir(output_path)

    rtk_x = pd.to_numeric(pd.Series(rtk_x), errors="coerce")
    rtk_y = pd.to_numeric(pd.Series(rtk_y), errors="coerce")
    uwb_x_aligned = pd.to_numeric(pd.Series(uwb_x_aligned), errors="coerce")
    uwb_y_aligned = pd.to_numeric(pd.Series(uwb_y_aligned), errors="coerce")

    plt.figure(figsize=(8, 8))

    m_rtk = rtk_x.notna() & rtk_y.notna()
    if m_rtk.any():
        plt.plot(rtk_x[m_rtk].to_numpy(), rtk_y[m_rtk].to_numpy(), label="RTK Scene Trajectory", linewidth=2.0)

    m_uwb = uwb_x_aligned.notna() & uwb_y_aligned.notna()
    if m_uwb.any():
        plt.plot(
            uwb_x_aligned[m_uwb].to_numpy(),
            uwb_y_aligned[m_uwb].to_numpy(),
            label="Aligned UWB Scene Trajectory",
            linewidth=1.5
        )

    plt.xlabel("Scene X (m)")
    plt.ylabel("Scene Y (m)")
    plt.title("Aligned UWB vs RTK in Parking-Lot Scene Coordinates")
    plt.axis("equal")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=FIG_DPI)
    plt.close()


# =========================================================
# 11. Single-file processing
# =========================================================
def process_one_file(
    name: str,
    uwb_input_path: Path,
    rtk_txt_path: Path,
    log_path: Path,
    scene_ref: Dict[str, Any],
) -> Dict[str, Any]:
    file_stem = uwb_input_path.stem
    output_xlsx = OUTPUT_DIR / f"{file_stem}_uwb_rtk_aligned_result.xlsx"
    output_plot = OUTPUT_DIR / f"{file_stem}_uwb_rtk_aligned_trajectory.png"

    if log_path.exists():
        log_path.unlink()

    log_print(log_path, "=" * 90)
    log_print(log_path, f"开始处理: {name}")
    log_print(log_path, f"UWB: {uwb_input_path}")
    log_print(log_path, f"RTK: {rtk_txt_path}")
    log_print(log_path, f"输出 Excel: {output_xlsx}")
    log_print(log_path, f"输出轨迹图: {output_plot}")

    if not uwb_input_path.exists():
        raise FileNotFoundError(f"UWB 文件不存在: {uwb_input_path}")
    if not rtk_txt_path.exists():
        raise FileNotFoundError(f"RTK 文件不存在: {rtk_txt_path}")

    uwb_t, df_uwb = robust_load_uwb_csv_4ranges(uwb_input_path)
    df_ranges = df_uwb[RANGE_COLS].copy()

    log_print(log_path, f"[UWB] Samples = {len(df_ranges)}, estimated rate = {infer_rate(uwb_t):.3f} Hz")
    log_print(log_path, "[DEBUG] UWB dtypes:")
    append_log(log_path, str(df_uwb.dtypes))
    log_print(log_path, "[DEBUG] UWB head:")
    append_log(log_path, str(df_uwb.head(3)))

    anchors_xy = np.array([ANCHORS_2D[i] for i in ANCHOR_IDS], dtype=float)

    uwb_x_stage1, uwb_y_stage1, cross_residual, uwb_source_stage1 = solve_stage1_with_weak_prior(
        t=uwb_t,
        df_ranges=df_ranges,
        anchors_xy=anchors_xy,
    )

    uwb_x_filled, uwb_y_filled, uwb_source_filled = fill_short_gaps_linear(
        t=uwb_t,
        x=uwb_x_stage1,
        y=uwb_y_stage1,
        source=uwb_source_stage1,
        max_gap_points=MAX_GAP_FILL_POINTS,
    )

    uwb_x_smooth, uwb_y_smooth = smooth_trajectory_stage2(
        x=uwb_x_filled,
        y=uwb_y_filled,
        source=uwb_source_filled,
        window=SMOOTH_WIN,
    )

    uwb_x_flip, uwb_y_flip = simple_flip_uwb(uwb_x_smooth, uwb_y_smooth, flip_x=UWB_FLIP_X)

    df_rtk = read_rtk_series_to_scene(rtk_txt_path, scene_ref=scene_ref)
    rtk_t = df_rtk["时间_s"].to_numpy(dtype=float)
    rtk_x_scene = df_rtk["RTK场景X_m"].to_numpy(dtype=float)
    rtk_y_scene = df_rtk["RTK场景Y_m"].to_numpy(dtype=float)

    log_print(log_path, f"[RTK] Samples = {len(df_rtk)}, estimated rate = {infer_rate(rtk_t):.3f} Hz")
    log_print(log_path, "[DEBUG] RTK dtypes:")
    append_log(log_path, str(df_rtk.dtypes))
    log_print(log_path, "[DEBUG] RTK head:")
    append_log(log_path, str(df_rtk.head(3)))

    best = search_best_time_shift_for_alignment(
        uwb_t=uwb_t,
        uwb_x=uwb_x_flip,
        uwb_y=uwb_y_flip,
        cross_residual=cross_residual,
        source=uwb_source_filled,
        rtk_t=rtk_t,
        rtk_x=rtk_x_scene,
        rtk_y=rtk_y_scene,
    )

    if best is None:
        raise ValueError(f"配准失败：{name} 没有找到可用时间偏移/空间配准结果。")

    log_print(
        log_path,
        f"[ALIGN] mode = {best['align_mode']}, points = {best['align_points']}, "
        f"time_shift = {best['time_shift_s']:.3f}, rmse = {best['rmse']:.4f}"
    )

    valid_range_count = np.isfinite(
        df_ranges[RANGE_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    ).sum(axis=1)

    df_out = pd.DataFrame({
        "点序号": np.arange(1, len(uwb_t) + 1),
        "时间_s": pd.to_numeric(pd.Series(uwb_t), errors="coerce"),
        "rangetime(ms)": pd.to_numeric(df_uwb["rangetime(ms)"], errors="coerce") if "rangetime(ms)" in df_uwb.columns else np.nan,

        "range0(m)": pd.to_numeric(df_ranges["range0(m)"], errors="coerce"),
        "range1(m)": pd.to_numeric(df_ranges["range1(m)"], errors="coerce"),
        "range2(m)": pd.to_numeric(df_ranges["range2(m)"], errors="coerce"),
        "range3(m)": pd.to_numeric(df_ranges["range3(m)"], errors="coerce"),

        "有效测距个数": valid_range_count,

        "第一阶段UWB_X_m": pd.to_numeric(pd.Series(uwb_x_stage1), errors="coerce"),
        "第一阶段UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_stage1), errors="coerce"),

        "补全后UWB_X_m": pd.to_numeric(pd.Series(uwb_x_filled), errors="coerce"),
        "补全后UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_filled), errors="coerce"),

        "平滑后UWB_X_m": pd.to_numeric(pd.Series(uwb_x_smooth), errors="coerce"),
        "平滑后UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_smooth), errors="coerce"),

        "翻转后UWB_X_m": pd.to_numeric(pd.Series(uwb_x_flip), errors="coerce"),
        "翻转后UWB_Y_m": pd.to_numeric(pd.Series(uwb_y_flip), errors="coerce"),

        "UWB位置来源": pd.Series(uwb_source_filled).astype(str),
        "三圆交叉残差_m": pd.to_numeric(pd.Series(cross_residual), errors="coerce"),

        "RTK插值场景X_m": pd.to_numeric(pd.Series(best["rtk_x_interp"]), errors="coerce"),
        "RTK插值场景Y_m": pd.to_numeric(pd.Series(best["rtk_y_interp"]), errors="coerce"),

        "UWB配准后场景X_m": pd.to_numeric(pd.Series(best["uwb_x_aligned"]), errors="coerce"),
        "UWB配准后场景Y_m": pd.to_numeric(pd.Series(best["uwb_y_aligned"]), errors="coerce"),

        "是否参与配准": pd.Series(best["align_mask"]).astype(int),
    })

    df_out["定位误差X_m"] = to_num_series(df_out["UWB配准后场景X_m"]) - to_num_series(df_out["RTK插值场景X_m"])
    df_out["定位误差Y_m"] = to_num_series(df_out["UWB配准后场景Y_m"]) - to_num_series(df_out["RTK插值场景Y_m"])
    df_out["定位误差_RTK_m"] = np.sqrt(df_out["定位误差X_m"] ** 2 + df_out["定位误差Y_m"] ** 2)

    err_s = to_num_series(df_out["定位误差_RTK_m"])
    res_s = to_num_series(df_out["三圆交叉残差_m"])

    valid_err = err_s.notna()
    valid_res = res_s.notna()

    source_counts = df_out["UWB位置来源"].astype(str).value_counts(dropna=False).to_dict()

    summary_row = {
        "数据组": name,
        "状态": "成功",
        "错误信息": "",
        "日志文件": str(log_path),
        "UWB输入文件": str(uwb_input_path),
        "RTK文件": str(rtk_txt_path),
        "总样本数": len(df_out),
        "可得到UWB位置点数": finite_count_2cols(df_out["UWB配准后场景X_m"], df_out["UWB配准后场景Y_m"]),
        "有RTK对应点数": finite_count_2cols(df_out["RTK插值场景X_m"], df_out["RTK插值场景Y_m"]),
        "可计算误差点数": int(valid_err.sum()),
        "最佳时间偏移_s": float(best["time_shift_s"]),
        "旋转角_deg": float(best["theta_deg"]),
        "平移X_m": float(best["t"][0]),
        "平移Y_m": float(best["t"][1]),
        "配准RMSE_m": float(best["rmse"]),
        "配准模式": str(best["align_mode"]),
        "配准点数": int(best["align_points"]),
        "三圆交叉残差均值_m": float(res_s[valid_res].mean()) if valid_res.any() else np.nan,
        "三圆交叉残差RMSE_m": float(np.sqrt((res_s[valid_res] ** 2).mean())) if valid_res.any() else np.nan,
        "定位误差_RTK均值_m": float(err_s[valid_err].mean()) if valid_err.any() else np.nan,
        "定位误差_RTK_RMSE_m": float(np.sqrt((err_s[valid_err] ** 2).mean())) if valid_err.any() else np.nan,
        "定位误差_RTK_95分位_m": float(err_s[valid_err].quantile(0.95)) if valid_err.any() else np.nan,
        "direct_4range点数": int(source_counts.get("direct_4range", 0)),
        "direct_3range点数": int(source_counts.get("direct_3range", 0)),
        "prior_2range点数": int(source_counts.get("prior_2range", 0)),
        "prior_1range点数": int(source_counts.get("prior_1range", 0)),
        "interp_gap点数": int(source_counts.get("interp_gap", 0)),
        "none_0range点数": int(source_counts.get("none_0range", 0)),
        "输出Excel": str(output_xlsx),
        "输出轨迹图": str(output_plot),
    }

    summary_df = pd.DataFrame([summary_row])

    with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
        df_out.to_excel(writer, sheet_name="逐点结果", index=False)
        summary_df.to_excel(writer, sheet_name="统计汇总", index=False)
        df_rtk.to_excel(writer, sheet_name="RTK原始转换结果", index=False)

    plot_aligned_trajectory(
        rtk_x=best["rtk_x_interp"],
        rtk_y=best["rtk_y_interp"],
        uwb_x_aligned=best["uwb_x_aligned"],
        uwb_y_aligned=best["uwb_y_aligned"],
        output_path=output_plot,
    )

    log_print(log_path, "Processing completed.")
    log_print(log_path, f"Best time shift (s): {best['time_shift_s']:.3f}")
    log_print(log_path, f"Alignment RMSE (m): {best['rmse']:.4f}")
    log_print(log_path, f"Excel saved to: {output_xlsx}")
    log_print(log_path, f"Plot saved to: {output_plot}")

    return summary_row


# =========================================================
# 12. Main
# =========================================================
def main():
    ensure_dir(OUTPUT_DIR)
    print(f"Output dir: {OUTPUT_DIR}")
    print(f"Batch summary: {OUTPUT_SUMMARY_XLSX}")

    scene_ref = build_global_parking_scene(DATA_PAIRS)

    scene_info_df = pd.DataFrame([{
        "全局原点纬度_deg": scene_ref["lat0"],
        "全局原点经度_deg": scene_ref["lon0"],
        "场景旋转角_deg": scene_ref["theta_scene_deg"],
        "场景平移X_m": scene_ref["shift_x"],
        "场景平移Y_m": scene_ref["shift_y"],
        "说明": "所有RTK已统一到该停车场场景坐标系，可直接用于后续分区与5x5网格划分",
    }])
    scene_info_df.to_excel(OUTPUT_SCENE_INFO_XLSX, index=False)

    all_summary_rows = []

    for item in DATA_PAIRS:
        log_path = OUTPUT_DIR / f"{item['name']}_run_log.txt"

        try:
            result = process_one_file(
                name=item["name"],
                uwb_input_path=item["uwb"],
                rtk_txt_path=item["rtk"],
                log_path=log_path,
                scene_ref=scene_ref,
            )
            all_summary_rows.append(result)

        except Exception as e:
            err_msg = str(e)
            tb = traceback.format_exc()

            print("=" * 90)
            print(f"[ERROR] 文件处理失败: {item['name']}")
            print(f"UWB: {item['uwb']}")
            print(f"RTK: {item['rtk']}")
            print(f"原因: {err_msg}")
            print(tb)

            append_log(log_path, "=" * 90)
            append_log(log_path, f"[ERROR] 文件处理失败: {item['name']}")
            append_log(log_path, f"UWB: {item['uwb']}")
            append_log(log_path, f"RTK: {item['rtk']}")
            append_log(log_path, f"原因: {err_msg}")
            append_log(log_path, tb)

            all_summary_rows.append({
                "数据组": item["name"],
                "状态": "失败",
                "错误信息": err_msg,
                "日志文件": str(log_path),
                "UWB输入文件": str(item["uwb"]),
                "RTK文件": str(item["rtk"]),
                "总样本数": np.nan,
                "可得到UWB位置点数": np.nan,
                "有RTK对应点数": np.nan,
                "可计算误差点数": np.nan,
                "最佳时间偏移_s": np.nan,
                "旋转角_deg": np.nan,
                "平移X_m": np.nan,
                "平移Y_m": np.nan,
                "配准RMSE_m": np.nan,
                "配准模式": "",
                "配准点数": np.nan,
                "三圆交叉残差均值_m": np.nan,
                "三圆交叉残差RMSE_m": np.nan,
                "定位误差_RTK均值_m": np.nan,
                "定位误差_RTK_RMSE_m": np.nan,
                "定位误差_RTK_95分位_m": np.nan,
                "direct_4range点数": np.nan,
                "direct_3range点数": np.nan,
                "prior_2range点数": np.nan,
                "prior_1range点数": np.nan,
                "interp_gap点数": np.nan,
                "none_0range点数": np.nan,
                "输出Excel": "",
                "输出轨迹图": "",
            })

    df_batch_summary = pd.DataFrame(all_summary_rows)
    df_batch_summary.to_excel(OUTPUT_SUMMARY_XLSX, index=False)

    print("=" * 90)
    print("全部处理完成。")
    print(f"批量汇总已保存到: {OUTPUT_SUMMARY_XLSX}")
    print(f"停车场统一场景参考已保存到: {OUTPUT_SCENE_INFO_XLSX}")


if __name__ == "__main__":
    main()


Output dir: D:\Desktop\missing_output\batch_output
Batch summary: D:\Desktop\missing_output\batch_output\uwb_rtk_batch_summary.xlsx
开始处理: 1试验
UWB: D:\Desktop\missing_output\1试验_missing.csv
RTK: D:\Desktop\paper2\数据\20260330 A20门口的UWB四节点定位-rtk定位实验\20260330\rtk\1.txt
输出 Excel: D:\Desktop\missing_output\batch_output\1试验_missing_uwb_rtk_aligned_result.xlsx
输出轨迹图: D:\Desktop\missing_output\batch_output\1试验_missing_uwb_rtk_aligned_trajectory.png
[UWB] Samples = 1435, estimated rate = 8.929 Hz
[DEBUG] UWB dtypes:
[DEBUG] UWB head:
[RTK] Samples = 2896, estimated rate = 10.000 Hz
[DEBUG] RTK dtypes:
[DEBUG] RTK head:
[ALIGN] mode = strict, points = 1424, time_shift = -10.300, rmse = 0.8143
Processing completed.
Best time shift (s): -10.300
Alignment RMSE (m): 0.8143
Excel saved to: D:\Desktop\missing_output\batch_output\1试验_missing_uwb_rtk_aligned_result.xlsx
Plot saved to: D:\Desktop\missing_output\batch_output\1试验_missing_uwb_rtk_aligned_trajectory.png
开始处理: 2圆形
UWB: D:\Desktop\missing_outpu